# 🎬 ZERO-API AI SHORT-FORM VIDEO FACTORY (Colab T4 Free Tier Edition)

### Production-Grade Automated 9:16 Vertical Video Production Pipeline
**Hardware Target:** Google Colab Free Tier (NVIDIA Tesla T4 ~14.56 GB VRAM, Python 3.13.16, PyTorch 2.11.0+cu130)  
**Cost:** **\$0.00 / Zero Paid APIs** (No OpenAI, No ElevenLabs, No Runway, No Claude, No Cloud Billing)  
**TTS Engine:** **Piper TTS** (CPU ONNX Inference, en_US-lessac-medium)  
**LLM Engine:** **Qwen/Qwen2.5-7B-Instruct** (4-bit NF4 Quantization, Sequential GPU Execution)  
**Visual Engine:** **ByteDance/SDXL-Lightning** (4-Step UNet @ 512×896 9:16 + 10-Motion Ken Burns FFmpeg)

---

### Pipeline Architecture & Execution Sequence:
1. **Hardware Preflight & Diagnostics:** System compatibility verification & memory guard
2. **Trend Discovery Engine:** Google Trends RSS + DuckDuckGo News (via `ddgs`) with full source attribution
3. **Research & Brief Engine:** Source-grounded fact extraction (no fabricated statistics)
4. **AI Script Engine (GPU):** `Qwen2.5-7B-Instruct` 4-bit NF4 with Pydantic validation (10 scenes, ~50s)
5. **Memory Purge:** Immediate LLM eviction and CUDA cache flushing (~100% VRAM recovered)
6. **Voice Engine (CPU):** `Piper TTS` (en_US-lessac-medium) + `espeak-ng` fallback (22.05kHz mono, 1.05x speed)
7. **Visual Engine (GPU):** `SDXL-Lightning` 4-step generation @ 512×896 + 10-motion dynamic Ken Burns
8. **Memory Purge:** Immediate diffusion pipeline eviction and CUDA cache flushing
9. **Subtitle Engine:** Burned-in animated ASS subtitles (DejaVuSans-Bold, 76% height, black outline)
10. **Audio & Video Editor:** FFmpeg audio ducking (-18 dB BGM under narration), smooth transitions
11. **SEO Deliverables:** Multi-platform captions & hashtags for YouTube Shorts, Reels, and TikTok
12. **Persistent Export:** Direct storage to Google Drive (`/content/drive/MyDrive/VideoFactory/`)


## CELL 1: Environment Setup and Google Drive Mounting
- Mounts Google Drive for permanent deliverable storage across Colab restarts
- Configures workspace directory structure for scenes, audio, images, scripts, models, and outputs
- Automatically falls back to local storage if Google Drive is not mounted


In [ ]:
# CELL 1: Environment Setup & Google Drive Mount
import os
import sys
import logging
from pathlib import Path

# Configure unified logging
LOG_FORMAT = "%(asctime)s [%(levelname)s] %(message)s"
logging.basicConfig(level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("VideoFactory")

# Google Drive Mount
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_MOUNTED = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists()
        print("✓ Google Drive mounted successfully.")
    except Exception as e:
        print(f"Notice: Google Drive mount skipped or unavailable ({e}). Using local runtime disk.")

# Directory Resolution
if DRIVE_MOUNTED:
    BASE_DIR = Path("/content/drive/MyDrive/VideoFactory")
else:
    BASE_DIR = Path("/content/VideoFactory")

TEMP_DIR = Path("/content/temp/VideoFactory")

DIRS = {
    "base": BASE_DIR,
    "outputs": BASE_DIR / "outputs",
    "scenes": BASE_DIR / "scenes",
    "audio": BASE_DIR / "audio",
    "images": BASE_DIR / "images",
    "scripts": BASE_DIR / "scripts",
    "logs": BASE_DIR / "logs",
    "models_piper": BASE_DIR / "models" / "piper",
    "temp": TEMP_DIR,
}

for name, p in DIRS.items():
    p.mkdir(parents=True, exist_ok=True)

# Persistent file logging
log_file = DIRS["logs"] / "factory.log"
file_handler = logging.FileHandler(log_file, encoding="utf-8")
file_handler.setFormatter(logging.Formatter(LOG_FORMAT))
logger.addHandler(file_handler)

print(f"✓ Video Factory Workspace initialized at: {BASE_DIR}")
print(f"✓ Piper Models Directory initialized at: {DIRS['models_piper']}")
print(f"✓ Temporary Scratchpad initialized at: {TEMP_DIR}")


Mounted at /content/drive
✓ Google Drive mounted successfully.
✓ Video Factory Workspace initialized at: /content/drive/MyDrive/VideoFactory
✓ Piper Models Directory initialized at: /content/drive/MyDrive/VideoFactory/models/piper
✓ Temporary Scratchpad initialized at: /content/temp/VideoFactory


## CELL 2: System and Python Dependency Installation (Python 3.13 Compatible)
- Installs critical system utilities: `ffmpeg`, `espeak-ng`, `fonts-dejavu-core`, `libasound2-dev`
- Installs compatible libraries for Python 3.13: `piper-tts`, `transformers`, `diffusers`, `bitsandbytes`, `ddgs`, `soundfile`
- **High-Performance CPU Inference:** Piper TTS is fully supported on Python 3.13 without heavy C-extensions
- **Preserves existing PyTorch & CUDA:** Does not reinstall or downgrade PyTorch 2.11+cu130


In [ ]:
# CELL 2: System and Python Dependency Installation

import sys
import subprocess
import warnings
import importlib.util

# Suppress Python deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print(f"Python Runtime: {sys.version.split()[0]}")

# 1. Install System Packages
print("\n[1/3] Installing system packages...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "ffmpeg",
        "espeak-ng",
        "fonts-dejavu-core",
        "libasound2-dev"
    ],
    check=True
)

print("✓ System packages installed.")

# 2. Preserve Existing PyTorch and CUDA Environment
print("\n[2/3] Inspecting existing PyTorch and CUDA...")

import torch

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {gpu.total_memory / (1024**3):.2f} GB")

# 3. Install Python Dependencies
print("\n[3/3] Installing Python dependencies...")

packages = [
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "diffusers>=0.31.0",
    "safetensors",
    "sentencepiece",
    "ddgs>=9.0.0",
    "feedparser>=6.0.0",
    "piper-tts",
    "soundfile>=0.12.1",
    "moviepy>=2.0.0",
    "pydantic>=2.0.0",
    "imageio-ffmpeg>=0.5.1",
    "Pillow>=10.0.0",
    "numpy>=1.24.0",
    "requests>=2.28.0",
    "tqdm>=4.65.0"
]

# Install OpenCV only if it is not already available
if importlib.util.find_spec("cv2") is None:
    packages.append("opencv-python-headless>=4.8.0")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + packages,
    check=True
)

print("✓ Python dependencies installed.")

# 4. Verify Critical Imports
print("\nVerifying installed dependencies...")

import transformers
import diffusers
import pydantic
import soundfile
import cv2
import feedparser
import moviepy
import piper

print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ Diffusers: {diffusers.__version__}")
print(f"✓ Pydantic: {pydantic.__version__}")
print(f"✓ SoundFile: {soundfile.__version__}")
print(f"✓ OpenCV: {cv2.__version__}")
print(f"✓ MoviePy: {moviepy.__version__}")
print("✓ DDGS: Available")
print("✓ Piper TTS: Available")
print("✓ FFmpeg: Installed")
print("✓ Dependency installation completed.")

Python Runtime: 3.13.16

[1/3] Installing system packages...
✓ System packages installed.

[2/3] Inspecting existing PyTorch and CUDA...
✓ PyTorch: 2.11.0+cu130
✓ CUDA Available: True
✓ GPU: Tesla T4
✓ VRAM: 14.56 GB

[3/3] Installing Python dependencies...
✓ Python dependencies installed.

Verifying installed dependencies...
✓ Transformers: 5.22.0
✓ Diffusers: 0.42.1
✓ Pydantic: 2.14.1
✓ SoundFile: 0.14.0
✓ OpenCV: 4.14.0
✓ MoviePy: 2.1.2
✓ DDGS: Available
✓ Piper TTS: Available
✓ FFmpeg: Installed
✓ Dependency installation completed.


## CELL 3: Hardware Diagnostics, Memory Guard & Preflight Verification
- Queries GPU cores, total and available VRAM on NVIDIA Tesla T4 (~14.56 GB)
- Implements strict GPU cache flushing (`clean_gpu_memory`) for sequential model execution
- Provides `run_preflight_system_checks()` to test CLI tools and package availability before loading large models


In [ ]:
# CELL 3: Hardware Diagnostics & Memory Management

import os
import sys
import gc
import shutil
import subprocess
import torch


def get_system_memory_info():
    """Get system RAM information without requiring psutil."""

    try:
        page_size = os.sysconf("SC_PAGE_SIZE")
        total_pages = os.sysconf("SC_PHYS_PAGES")
        available_pages = os.sysconf("SC_AVPHYS_PAGES")

        total_ram = page_size * total_pages
        available_ram = page_size * available_pages

        return {
            "total": round(total_ram / (1024**3), 2),
            "available": round(available_ram / (1024**3), 2)
        }

    except (ValueError, OSError, AttributeError):
        return {
            "total": 0.0,
            "available": 0.0
        }


def check_hardware_diagnostics():

    diagnostics = {
        "python_version": sys.version.split()[0],
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": (
            torch.version.cuda if torch.cuda.is_available() else None
        ),
        "gpu_name": "None",
        "total_vram_gb": 0.0,
        "free_vram_gb": 0.0,
        "system_ram_gb": 0.0,
        "free_system_ram_gb": 0.0,
        "free_disk_gb": 0.0,
        "recommended_engine": "sdxl",
        "warnings": []
    }

    # GPU Diagnostics
    if diagnostics["cuda_available"]:

        try:
            gpu_properties = torch.cuda.get_device_properties(0)

            diagnostics["gpu_name"] = torch.cuda.get_device_name(0)

            diagnostics["total_vram_gb"] = round(
                gpu_properties.total_memory / (1024**3), 2
            )

            free_memory, total_memory = torch.cuda.mem_get_info(0)

            diagnostics["free_vram_gb"] = round(
                free_memory / (1024**3), 2
            )

        except RuntimeError as e:
            diagnostics["warnings"].append(
                f"GPU memory inspection failed: {e}"
            )

    else:
        diagnostics["recommended_engine"] = "procedural_fallback"
        diagnostics["warnings"].append(
            "No CUDA GPU detected. GPU-based image generation is unavailable."
        )

    # System RAM Diagnostics
    ram_info = get_system_memory_info()

    diagnostics["system_ram_gb"] = ram_info["total"]
    diagnostics["free_system_ram_gb"] = ram_info["available"]

    # Disk Diagnostics
    try:
        _, _, free_disk = shutil.disk_usage(os.getcwd())

        diagnostics["free_disk_gb"] = round(
            free_disk / (1024**3), 2
        )

    except OSError as e:
        diagnostics["warnings"].append(
            f"Disk inspection failed: {e}"
        )

    # Resource Warnings
    if diagnostics["cuda_available"]:

        if diagnostics["free_vram_gb"] < 8.0:
            diagnostics["warnings"].append(
                "Low available VRAM. SDXL generation may fail. "
                "Use sequential model loading and memory cleanup."
            )

        if diagnostics["free_vram_gb"] < 5.0:
            diagnostics["recommended_engine"] = "ken_burns"
            diagnostics["warnings"].append(
                "Very limited free VRAM. Prefer procedural visuals "
                "with Ken Burns animation."
            )

    if diagnostics["free_system_ram_gb"] < 2.0:
        diagnostics["warnings"].append(
            "Low available system RAM. Avoid loading multiple models simultaneously."
        )

    if diagnostics["free_disk_gb"] < 5.0:
        diagnostics["warnings"].append(
            "Low disk space. Clean temporary files before generation."
        )

    # Print Diagnostics
    print("=" * 65)
    print("           COLAB HARDWARE & RUNTIME DIAGNOSTICS")
    print("=" * 65)

    print(f" Python Version      : {diagnostics['python_version']}")
    print(f" PyTorch Version     : {diagnostics['pytorch_version']}")
    print(f" CUDA Available      : {diagnostics['cuda_available']}")

    if diagnostics["cuda_available"]:
        print(f" GPU Model           : {diagnostics['gpu_name']}")
        print(f" Total VRAM          : {diagnostics['total_vram_gb']} GB")
        print(f" Available VRAM      : {diagnostics['free_vram_gb']} GB")
        print(f" CUDA Version        : {diagnostics['cuda_version']}")

    print(f" Host RAM (Total)    : {diagnostics['system_ram_gb']} GB")
    print(f" Host RAM (Available): {diagnostics['free_system_ram_gb']} GB")
    print(f" Available Disk      : {diagnostics['free_disk_gb']} GB")

    print(
        f" Recommended Engine  : "
        f"{diagnostics['recommended_engine'].upper()}"
    )

    if diagnostics["warnings"]:
        for warning in diagnostics["warnings"]:
            print(f"  [!] {warning}")

    print("=" * 65)

    return diagnostics


def run_preflight_system_checks():

    results = {}

    # System Executables
    for tool in ["ffmpeg", "ffprobe", "espeak-ng"]:
        results[tool] = shutil.which(tool) is not None

    # Piper: Check CLI and Python module
    piper_cli = shutil.which("piper") is not None

    try:
        import piper
        piper_module = True
    except ImportError:
        piper_module = False

    results["piper_cli"] = piper_cli
    results["piper"] = piper_cli or piper_module

    # DDGS
    try:
        from ddgs import DDGS
        results["ddgs"] = True
    except ImportError:
        results["ddgs"] = False

    return results


def get_gpu_memory_info() -> str:

    if not torch.cuda.is_available():
        return "CPU Mode (CUDA unavailable)"

    try:
        allocated = torch.cuda.memory_allocated(0) / (1024**3)
        reserved = torch.cuda.memory_reserved(0) / (1024**3)

        free_memory, total_memory = torch.cuda.mem_get_info(0)

        free_gb = free_memory / (1024**3)
        total_gb = total_memory / (1024**3)

        return (
            f"Allocated: {allocated:.2f} GB | "
            f"Reserved: {reserved:.2f} GB | "
            f"Free: {free_gb:.2f} GB | "
            f"Total: {total_gb:.2f} GB"
        )

    except RuntimeError as e:
        return f"GPU memory information unavailable: {e}"


def clean_gpu_memory(stage_name: str = "Stage"):

    mem_before = get_gpu_memory_info()

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    mem_after = get_gpu_memory_info()

    logger.info(
        f"[{stage_name}] Memory cleanup completed. "
        f"Before: [{mem_before}] -> After: [{mem_after}]"
    )


# Initialize diagnostics
hardware_info = check_hardware_diagnostics()

# Initialize preflight checks
preflight = run_preflight_system_checks()

print("\nSYSTEM PREFLIGHT RESULTS")

for component, available in preflight.items():
    status = "AVAILABLE" if available else "NOT AVAILABLE"
    print(f" {component:<12}: {status}")

           COLAB HARDWARE & RUNTIME DIAGNOSTICS
 Python Version      : 3.13.16
 PyTorch Version     : 2.11.0+cu130
 CUDA Available      : True
 GPU Model           : Tesla T4
 Total VRAM          : 14.56 GB
 Available VRAM      : 14.46 GB
 CUDA Version        : 13.0
 Host RAM (Total)    : 12.67 GB
 Host RAM (Available): 4.48 GB
 Available Disk      : 112.09 GB
 Recommended Engine  : SDXL

SYSTEM PREFLIGHT RESULTS
 ffmpeg      : AVAILABLE
 ffprobe     : AVAILABLE
 espeak-ng   : AVAILABLE
 piper_cli   : AVAILABLE
 piper       : AVAILABLE
 ddgs        : AVAILABLE


## CELL 4: Zero-API Trend Discovery Engine (Repaired & Resilient)
- Uses `ddgs` as the primary search client with compatible fallback
- Queries Google Trends public RSS feed (`https://trends.google.com/trending/rss?geo={geo}`)
- **Repaired:** Correctly uses `enumerate(collected[:max_stories], start=1)` (eliminates `NameError`)
- Deduplicates topics and preserves source name, publication date, headline, summary, and article URL
- **Strict Grounding:** Clearly labels `[VERIFIED REAL-TIME NEWS]` vs `[CURATED TECHNICAL FALLBACK]`


In [ ]:
# CELL 4: Trend Discovery Engine

import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from typing import List, Dict, Any


# Safe fallback topics.
# These are topic suggestions, NOT claims about current events.

CURATED_FALLBACK_TOPICS = {
    "Artificial Intelligence and Robotics": [
        {
            "headline": "How Small Language Models Are Changing On-Device AI",
            "description": (
                "Explore quantized language models, local inference, "
                "memory optimization, and practical edge AI applications."
            ),
            "source": "Curated Educational Topic",
            "pub_date": "Not applicable",
            "url": "",
            "is_live_news": False
        },
        {
            "headline": "How Humanoid Robots Understand and Interact With the World",
            "description": (
                "Explore computer vision, motion planning, reinforcement "
                "learning, and robotic control systems."
            ),
            "source": "Curated Educational Topic",
            "pub_date": "Not applicable",
            "url": "",
            "is_live_news": False
        }
    ],

    "Default": [
        {
            "headline": "Emerging Technologies Transforming Everyday Life",
            "description": (
                "Explore recent technological developments, their "
                "underlying engineering principles, and practical applications."
            ),
            "source": "Curated Educational Topic",
            "pub_date": "Not applicable",
            "url": "",
            "is_live_news": False
        }
    ]
}


def fetch_google_trends_rss(
    geo: str = "IN",
    max_items: int = 8
) -> List[Dict[str, Any]]:
    """
    Fetch trending searches from Google Trends RSS.
    """

    url = (
        "https://trends.google.com/trending/rss?"
        + urllib.parse.urlencode({"geo": geo})
    )

    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    trends = []

    try:
        with urllib.request.urlopen(req, timeout=15) as response:

            content = response.read()
            root = ET.fromstring(content)

            namespace = {
                "ht": "https://trends.google.com/trending/rss"
            }

            for item in root.findall(".//item"):

                title = item.findtext("title", default="").strip()

                traffic = item.findtext(
                    "ht:approx_traffic",
                    default="Trending",
                    namespaces=namespace
                )

                pub_date = item.findtext(
                    "pubDate",
                    default="",
                    namespaces=namespace
                )

                snippets = []

                for news_item in item.findall("ht:news_item", namespace):

                    story_title = news_item.findtext(
                        "ht:news_item_title",
                        default="",
                        namespaces=namespace
                    ).strip()

                    story_source = news_item.findtext(
                        "ht:news_item_source",
                        default="Google Trends",
                        namespaces=namespace
                    ).strip()

                    story_url = news_item.findtext(
                        "ht:news_item_url",
                        default="",
                        namespaces=namespace
                    ).strip()

                    if story_title:

                        snippets.append({
                            "title": story_title,
                            "source": story_source,
                            "url": story_url
                        })

                if title:

                    trends.append({
                        "query": title,
                        "traffic": traffic,
                        "pub_date": pub_date,
                        "snippets": snippets
                    })

                if len(trends) >= max_items:
                    break

    except Exception as e:

        logger.warning(
            f"Google Trends RSS retrieval failed: {e}"
        )

    return trends


def search_ddg_news(
    query: str,
    max_results: int = 5
) -> List[Dict[str, Any]]:
    """
    Retrieve news search results using the DDGS package.

    Search results are treated as retrieved information,
    not independently verified reporting.
    """

    articles = []

    try:

        from ddgs import DDGS

        with DDGS() as ddgs:

            results = ddgs.news(
                query,
                max_results=max_results
            )

            for result in results:

                title = str(
                    result.get("title") or ""
                ).strip()

                description = str(
                    result.get("body") or ""
                ).strip()

                source = str(
                    result.get("source") or "Unknown"
                ).strip()

                pub_date = str(
                    result.get("date") or "Date unavailable"
                ).strip()

                url = str(
                    result.get("url") or ""
                ).strip()

                if not title:
                    continue

                articles.append({
                    "headline": title,
                    "description": description,
                    "source": source,
                    "pub_date": pub_date,
                    "url": url,
                    "is_live_news": True
                })

    except Exception as e:

        logger.warning(
            f"DuckDuckGo News retrieval failed: {e}"
        )

    return articles


def fetch_live_trend_context(
    niche: str = "Artificial Intelligence and Robotics",
    geo: str = "IN",
    max_stories: int = 5
) -> str:
    """
    Combine Google Trends and DDGS news results.

    If live retrieval fails, use clearly labelled educational topics.
    """

    if max_stories < 1:
        raise ValueError("max_stories must be at least 1.")

    google_trends = fetch_google_trends_rss(
        geo=geo,
        max_items=8
    )

    niche_news = search_ddg_news(
        query=f"{niche} breakthrough OR development",
        max_results=max_stories
    )

    collected = []
    seen = set()

    # 1. Collect DuckDuckGo News results
    for item in niche_news:

        key = item["headline"].lower().strip()

        if key and key not in seen:

            seen.add(key)
            collected.append(item)

    # 2. Supplement with Google Trends news snippets
    if len(collected) < max_stories:

        for trend in google_trends:

            for snippet in trend.get("snippets", []):

                headline = snippet.get("title", "").strip()
                key = headline.lower()

                if not headline or key in seen:
                    continue

                seen.add(key)

                collected.append({
                    "headline": headline,
                    "description": (
                        f"Associated Google Trends search: "
                        f"{trend['query']}. "
                        f"Search interest: {trend['traffic']}."
                    ),
                    "source": snippet.get(
                        "source",
                        "Google Trends"
                    ),
                    "pub_date": trend.get(
                        "pub_date",
                        "Date unavailable"
                    ),
                    "url": snippet.get("url", ""),
                    "is_live_news": True
                })

                if len(collected) >= max_stories:
                    break

            if len(collected) >= max_stories:
                break

    # 3. Use safe educational fallback topics
    using_fallback = False

    if not collected:

        using_fallback = True

        logger.warning(
            "No current news retrieved. Using educational fallback topics."
        )

        fallback_list = CURATED_FALLBACK_TOPICS.get(
            niche,
            CURATED_FALLBACK_TOPICS["Default"]
        )

        collected.extend(
            fallback_list[:max_stories]
        )

    # 4. Build context for the script-generation engine
    if using_fallback:

        header_status = (
            "[NO CURRENT NEWS RETRIEVED - "
            "EDUCATIONAL FALLBACK]"
        )

    else:

        header_status = (
            "[NEWS SEARCH RESULTS - "
            "NOT INDEPENDENTLY VERIFIED]"
        )

    lines = [
        f"=== TREND CONTEXT: {niche.upper()} ===",
        f"REGION: {geo}",
        f"STATUS: {header_status}"
    ]

    if google_trends:

        trend_strings = [
            f"{trend['query']} ({trend['traffic']})"
            for trend in google_trends[:5]
        ]

        lines.append(
            "Google Trends Queries: "
            + ", ".join(trend_strings)
        )

    lines.append("\nRetrieved Stories and Topics:")

    for index, story in enumerate(
        collected[:max_stories],
        start=1
    ):

        if story.get("is_live_news", False):
            tag = "[SEARCH-RETRIEVED NEWS]"
        else:
            tag = "[EDUCATIONAL FALLBACK]"

        lines.append(
            f"\n[{index}] {tag} {story['headline']}"
        )

        lines.append(
            f"    Source: {story.get('source', 'Unknown')}"
        )

        lines.append(
            f"    Date: {story.get('pub_date', 'Unknown')}"
        )

        if story.get("url"):
            lines.append(
                f"    URL: {story['url']}"
            )

        lines.append(
            f"    Summary: {story.get('description', '')}"
        )

    lines.append(
        "\nIMPORTANT: Search-retrieved information is not "
        "independently verified. Do not invent dates, statistics, "
        "quotes, events, or factual claims. Educational fallback "
        "topics must not be presented as breaking news."
    )

    return "\n".join(lines)


# Initialize trend discovery
sample_trends = fetch_live_trend_context(
    niche="Artificial Intelligence and Robotics",
    geo="IN",
    max_stories=3
)

print("Trend Discovery Engine initialized.")
print("\nRetrieved context:\n")
print(sample_trends)

Trend Discovery Engine initialized.

Retrieved context:

=== TREND CONTEXT: ARTIFICIAL INTELLIGENCE AND ROBOTICS ===
REGION: IN
STATUS: [NEWS SEARCH RESULTS - NOT INDEPENDENTLY VERIFIED]
Google Trends Queries: వరి (500+), temple (200+), asteroid (500+), பிக் பாஸ் தமிழ் (200+), ಕಬ್ಬು (1000+)

Retrieved Stories and Topics:

[1] [SEARCH-RETRIEVED NEWS] Ninth Annual AI Breakthrough Awards Program Recognizes the World's Most Innovative Artificial Intelligence Technologies and Companies
    Source: Yahoo Finance
    Date: 2026-06-25T13:59:00+00:00
    URL: https://finance.yahoo.com/technology/ai/articles/ninth-annual-ai-breakthrough-awards-140000624.html
    Summary: LOS ANGELES, June 25, 2026 (GLOBE NEWSWIRE) -- AI Breakthrough, a leading market intelligence organization that recognizes the top companies, technologies and products in the global Artificial Intelligence (AI) market, today announced the results of its ...

[2] [SEARCH-RETRIEVED NEWS] Analytics8 Wins "Data Management Innovation

## CELL 5: Source-Grounded Topic Research Engine
- Extracts verified facts directly from retrieved headlines and descriptions
- **No Hallucinated Claims:** Does not invent generic 10x speedups or unverified statistics
- Formulates a source-grounded research brief with publication date, source attribution, and verification URL


In [ ]:
# CELL 5: Source-Grounded Topic Research Engine

from typing import Dict, Any, List
from urllib.parse import urlparse
import re


def evaluate_and_select_topic(
    trend_context: str,
    niche: str
) -> Dict[str, Any]:
    """
    Parse trend context and select a topic using available evidence.

    Important:
    - Search results are not independently verified.
    - Curated fallback topics are not current news.
    - No unsupported statistics or factual claims are generated.
    """

    if not isinstance(trend_context, str) or not trend_context.strip():
        raise ValueError("Trend context is empty or invalid.")

    parsed_stories: List[Dict[str, Any]] = []

    current_story = None
    current_tag = None
    status_line = ""

    # 1. Parse trend context
    for line in trend_context.splitlines():

        line_s = line.strip()

        if line_s.startswith("STATUS:"):
            status_line = line_s
            continue

        # Identify story headings
        match = re.match(
            r"^\[(\d+)\]\s+\[(SEARCH-RETRIEVED NEWS|EDUCATIONAL FALLBACK)\]\s+(.+)$",
            line_s
        )

        if match:

            if current_story and current_story.get("headline"):
                parsed_stories.append(current_story)

            current_tag = match.group(2)

            current_story = {
                "headline": match.group(3).strip(),
                "source": "Unknown",
                "date": "Date unavailable",
                "url": "",
                "summary": "",
                "is_live_news": (
                    current_tag == "SEARCH-RETRIEVED NEWS"
                ),
                "is_fallback": (
                    current_tag == "EDUCATIONAL FALLBACK"
                )
            }

            continue

        if current_story is None:
            continue

        # Parse source and publication date
        if line_s.startswith("Source:"):

            parts = line_s.split("|")

            current_story["source"] = (
                parts[0].replace("Source:", "").strip()
            )

            if len(parts) > 1 and "Date:" in parts[1]:

                current_story["date"] = (
                    parts[1].replace("Date:", "").strip()
                )

        elif line_s.startswith("Date:"):

            current_story["date"] = (
                line_s.replace("Date:", "").strip()
            )

        elif line_s.startswith("URL:"):

            current_story["url"] = (
                line_s.replace("URL:", "").strip()
            )

        elif line_s.startswith("Summary:"):

            current_story["summary"] = (
                line_s.replace("Summary:", "").strip()
            )

    if current_story and current_story.get("headline"):
        parsed_stories.append(current_story)

    # 2. Identify whether the context contains fallback topics
    context_is_fallback = (
        "EDUCATIONAL FALLBACK" in status_line
        or "NO CURRENT NEWS RETRIEVED" in status_line
    )

    # 3. Select the best available story
    if parsed_stories:

        # Prefer search-retrieved stories when available.
        live_stories = [
            story for story in parsed_stories
            if story["is_live_news"]
        ]

        candidates = live_stories if live_stories else parsed_stories

        # Prefer stories with a description and a usable source URL.
        best_story = max(
            candidates,
            key=lambda story: (
                bool(story.get("summary")),
                bool(story.get("url")),
                len(story.get("summary", ""))
            )
        )

    else:

        # Safe fallback if the input cannot be parsed.
        best_story = {
            "headline": f"Emerging Technologies in {niche}",
            "source": "Curated Educational Topic",
            "date": "Not applicable",
            "url": "",
            "summary": (
                f"An educational overview of important concepts "
                f"and engineering developments in {niche}."
            ),
            "is_live_news": False,
            "is_fallback": True
        }

        context_is_fallback = True

    # 4. Extract selected information
    headline = best_story["headline"]
    source = best_story.get("source", "Unknown")
    date = best_story.get("date", "Date unavailable")
    url = best_story.get("url", "")
    summary = best_story.get("summary", "")

    is_live_news = best_story.get("is_live_news", False)
    is_fallback = best_story.get("is_fallback", False)

    # 5. Validate source URL format
    valid_source_url = False

    if url:

        try:
            parsed_url = urlparse(url)

            valid_source_url = (
                parsed_url.scheme in ("http", "https")
                and bool(parsed_url.netloc)
            )

        except Exception:
            valid_source_url = False

    # 6. Establish evidence status
    if is_live_news and not context_is_fallback:

        evidence_status = (
            "Search-Retrieved News - Not Independently Verified"
        )

    else:

        evidence_status = (
            "Curated Educational Reference - Not Current News"
        )

    # 7. Compile source-grounded facts
    key_facts = []

    if summary:
        key_facts.append(
            f"Retrieved description: {summary}"
        )

    key_facts.append(
        f"Source: {source}"
    )

    key_facts.append(
        f"Publication date: {date}"
    )

    if valid_source_url:

        key_facts.append(
            f"Source URL: {url}"
        )

    key_facts.append(
        "Evidence limitation: The retrieved information has "
        "not been independently fact-checked."
    )

    if not is_live_news:

        key_facts.append(
            "This is an educational topic, not a verified current event."
        )

    # 8. Create research brief
    research_brief = {

        "selected_topic": headline,

        "niche": niche,

        "evidence_status": evidence_status,

        "is_live_news": is_live_news,

        "is_fallback": is_fallback or context_is_fallback,

        "primary_source": source,

        "publication_date": date,

        "source_url": url if valid_source_url else "",

        "source_url_valid": valid_source_url,

        "summary": summary,

        "why_it_matters": (
            "Explain the relevance of this topic using only "
            "information supported by the retrieved source."
        ),

        "key_facts": key_facts,

        "narrative_hook": (
            f"What should people understand about {headline}? "
            "Let's examine the available evidence."
        ),

        "call_to_action": (
            "Follow for more technology explanations and "
            "source-grounded updates."
        ),

        "source_grounding_rules": [

            "Do not invent statistics or performance claims.",

            "Do not invent publication dates or quotations.",

            "Do not present educational fallback topics as breaking news.",

            "Do not claim independent verification.",

            "Distinguish reported information from interpretation.",

            "Preserve the original source URL when available."
        ]
    }

    print(
        f"✓ Research Brief compiled: "
        f"{research_brief['selected_topic']}"
    )

    print(
        f"  Evidence Status: "
        f"{research_brief['evidence_status']}"
    )

    print(
        f"  Primary Source: {research_brief['primary_source']}"
    )

    print(
        f"  Source URL Available: "
        f"{research_brief['source_url_valid']}"
    )

    return research_brief


# Generate initial research brief
sample_brief = evaluate_and_select_topic(
    sample_trends,
    "Artificial Intelligence and Robotics"
)

✓ Research Brief compiled: Ninth Annual AI Breakthrough Awards Program Recognizes the World's Most Innovative Artificial Intelligence Technologies and Companies
  Evidence Status: Search-Retrieved News - Not Independently Verified
  Primary Source: Yahoo Finance
  Source URL Available: True


## CELL 6: AI Brain — Script and Scene Generation (Qwen2.5-7B NF4)
- Loads `Qwen/Qwen2.5-7B-Instruct` in 4-bit NF4 quantization (`load_in_4bit=True`, `torch.float16`)
- Strict Pydantic validation:
  - Exactly 10 scenes, sequential scene numbers 1..10
  - Duration ~50s total (~5.0s per scene)
  - Narration (~11–14 words per scene) strictly grounded in the research brief
  - Subtitles (~4–7 words)
  - Cinematic visual prompts specifying lighting, camera angle, and style (no text in images)
- **CRITICAL:** Explicitly deletes model and tokenizer, executes garbage collection, and flushes CUDA cache immediately after generation


In [ ]:

# ============================================================
# CELL 6: AI BRAIN - MEMORY-OPTIMIZED SCRIPT GENERATION
# ============================================================

import json
import re
import gc
import os
import torch

from typing import List, Dict, Any
from pydantic import (
    BaseModel,
    Field,
    field_validator,
    model_validator
)


# ============================================================
# 1. RAM MANAGEMENT
# ============================================================

MIN_RAM_FOR_QWEN_GB = 6.5


def get_available_system_ram_gb():

    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")

        return (pages * page_size) / (1024 ** 3)

    except Exception:
        return None


def release_llm_memory():

    gc.collect()

    if torch.cuda.is_available():

        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

    gc.collect()

    try:
        clean_gpu_memory("Qwen model cleanup")
    except Exception:
        pass


# ============================================================
# 2. STRUCTURED OUTPUT SCHEMAS
# ============================================================

class Scene(BaseModel):

    scene_number: int = Field(..., ge=1, le=10)

    visual_prompt: str = Field(..., min_length=15)

    voiceover_text: str = Field(..., min_length=10)

    subtitle_text: str = Field(..., min_length=4)

    duration_seconds: float = Field(
        default=5.0,
        ge=5.0,
        le=5.0
    )

    @field_validator(
        "visual_prompt",
        "voiceover_text",
        "subtitle_text"
    )
    @classmethod
    def clean_text(cls, value: str):

        value = value.strip()

        if not value:
            raise ValueError("Scene text cannot be empty.")

        return value


class VideoPlan(BaseModel):

    selected_topic: str = Field(..., min_length=5)

    hook_title: str = Field(..., min_length=5)

    target_audience: str = Field(
        default="Tech enthusiasts, developers, and curious minds"
    )

    scenes: List[Scene] = Field(
        ...,
        min_length=10,
        max_length=10
    )

    social_caption: str = Field(..., min_length=15)

    hashtags: List[str] = Field(..., min_length=4)

    @model_validator(mode="after")
    def validate_video_plan(self):

        if len(self.scenes) != 10:
            raise ValueError(
                "Video must contain exactly 10 scenes."
            )

        for index, scene in enumerate(self.scenes, start=1):

            if scene.scene_number != index:
                raise ValueError(
                    f"Expected scene {index}, "
                    f"received {scene.scene_number}."
                )

        total_duration = sum(
            scene.duration_seconds for scene in self.scenes
        )

        if total_duration != 50.0:
            raise ValueError(
                f"Expected 50 seconds, got {total_duration}."
            )

        return self

    @field_validator("hashtags")
    @classmethod
    def validate_hashtags(cls, hashtags: List[str]):

        cleaned = []

        for tag in hashtags:

            tag = tag.strip()

            if not tag:
                continue

            if not tag.startswith("#"):
                tag = "#" + tag

            cleaned.append(tag)

        if len(cleaned) < 4:
            raise ValueError(
                "At least four valid hashtags are required."
            )

        return cleaned


# ============================================================
# 3. GROUNDED FALLBACK SCRIPT GENERATOR
# ============================================================

def generate_fallback_script(
    brief: Dict[str, Any]
) -> VideoPlan:

    topic = str(
        brief.get("selected_topic", "Emerging Technology")
    ).strip()

    source = str(
        brief.get("primary_source", "Unknown")
    ).strip()

    summary = str(
        brief.get("summary", "")
    ).strip()

    evidence_status = str(
        brief.get("evidence_status", "Not specified")
    )

    is_live_news = bool(
        brief.get("is_live_news", False)
    )

    is_fallback = bool(
        brief.get("is_fallback", False)
    )

    if is_fallback or not is_live_news:

        opening = (
            f"Today, we are exploring {topic[:65]} "
            "and the technology behind it."
        )

        attribution = (
            "This video is an educational overview, "
            "not a breaking news report."
        )

    else:

        opening = (
            f"Here is a closer look at {topic[:65]} "
            "and the information currently available."
        )

        attribution = (
            f"The available information comes from {source}. "
            "It has not been independently verified."
        )

    if summary:

        summary_text = summary[:350].strip()

    else:

        summary_text = (
            f"The available research context identifies "
            f"{topic} as its main subject."
        )

    voiceovers = [

        opening,

        f"First, let's understand what {topic[:65]} actually means.",

        attribution,

        f"The available material describes the following: {summary_text}",

        "It is important to separate documented information from assumptions.",

        "The available source does not establish every possible technical detail.",

        "Understanding the underlying concepts helps put this subject into context.",

        "Further evidence is needed before drawing broader technical conclusions.",

        f"The central question remains: what can {topic[:65]} eventually enable?",

        "What is your perspective? Share your thoughts and follow for more."
    ]

    subtitles = [

        "Understanding The Topic",

        "Start With The Basics",

        "What The Source Says",

        "Available Research Details",

        "Facts Versus Assumptions",

        "Understanding The Limitations",

        "The Technical Context",

        "More Evidence Is Needed",

        "What Comes Next?",

        "Share Your Perspective"
    ]

    visual_prompts = [

        "Cinematic macro visualization of an advanced technological concept, dramatic lighting, realistic materials, vertical composition",

        "Minimal futuristic laboratory environment with abstract technical equipment, cinematic depth, vertical composition",

        "Close-up conceptual visualization of research documentation and scientific equipment, no readable text, cinematic lighting",

        "Abstract scientific visualization representing information flow and interconnected technical concepts, detailed cinematic composition",

        "Conceptual split composition showing established engineering principles and uncertain possibilities, realistic lighting",

        "Dark cinematic research environment with abstract technical diagrams and subtle glowing elements, no readable text",

        "Detailed macro visualization of interconnected technological components, realistic textures, shallow depth of field",

        "Scientific research environment with engineers examining abstract visual data, cinematic realism",

        "Conceptual futuristic technology visualization representing potential applications, realistic cinematic atmosphere",

        "Minimal cinematic closing shot of a futuristic horizon with subtle technological elements, vertical composition"
    ]

    scenes = []

    for index in range(10):

        scenes.append(
            Scene(
                scene_number=index + 1,
                visual_prompt=visual_prompts[index],
                voiceover_text=voiceovers[index],
                subtitle_text=subtitles[index],
                duration_seconds=5.0
            )
        )

    caption_prefix = (
        "Explore this educational breakdown"
        if is_fallback or not is_live_news
        else "Explore this source-grounded report"
    )

    social_caption = (
        f"{caption_prefix}: {topic}. "
        f"Evidence status: {evidence_status}. "
        "Separate documented information from speculation. "
        "Share your thoughts below."
    )

    return VideoPlan(

        selected_topic=topic,

        hook_title=f"Understanding {topic[:45]}",

        target_audience=(
            "Tech enthusiasts, developers, students, and curious minds"
        ),

        scenes=scenes,

        social_caption=social_caption,

        hashtags=[
            "#Technology",
            "#ArtificialIntelligence",
            "#Innovation",
            "#Engineering",
            "#FutureTech",
            "#TechExplained"
        ]
    )


# ============================================================
# 4. JSON EXTRACTION
# ============================================================

def extract_json_from_response(text: str) -> Dict[str, Any]:

    if not text or not text.strip():
        raise ValueError("LLM returned an empty response.")

    cleaned = text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No valid JSON object found.")

    parsed = json.loads(cleaned[start:end + 1])

    if not isinstance(parsed, dict):
        raise ValueError("Expected a JSON object.")

    return parsed


# ============================================================
# 5. QWEN 2.5 - 7B INSTRUCT
# ============================================================

def generate_video_script_llm(
    brief: Dict[str, Any],
    model_id: str = "Qwen/Qwen2.5-7B-Instruct"
) -> VideoPlan:

    # --------------------------------------------------------
    # Check available resources before loading the model
    # --------------------------------------------------------

    if not torch.cuda.is_available():

        print("CUDA unavailable. Using grounded fallback.")

        return generate_fallback_script(brief)

    available_ram = get_available_system_ram_gb()

    if available_ram is None:

        print(
            "Cannot verify available system RAM. "
            "Skipping Qwen to avoid an uncontrolled memory failure."
        )

        return generate_fallback_script(brief)

    print(
        f"Available system RAM before Qwen: "
        f"{available_ram:.2f} GB"
    )

    if available_ram < MIN_RAM_FOR_QWEN_GB:

        print(
            f"Insufficient RAM for Qwen. "
            f"Required: {MIN_RAM_FOR_QWEN_GB:.1f} GB."
        )

        print("Using grounded fallback script instead.")

        return generate_fallback_script(brief)

    # --------------------------------------------------------
    # Initialize all temporary model objects
    # --------------------------------------------------------

    model = None
    tokenizer = None
    inputs = None
    outputs = None
    generated_tokens = None

    prompt_text = None
    output_text = None
    parsed_json = None

    plan = None

    try:

        from transformers import (
            AutoModelForCausalLM,
            AutoTokenizer,
            BitsAndBytesConfig
        )

        print(f"\nLoading {model_id} in 4-bit NF4 mode...")

        quantization_config = BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_use_double_quant=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_compute_dtype=torch.float16
        )

        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=False
        )

        # Place the quantized model on the GPU instead of
        # allowing automatic CPU offloading.
        model = AutoModelForCausalLM.from_pretrained(

            model_id,

            quantization_config=quantization_config,

            device_map={"": 0},

            trust_remote_code=False,

            low_cpu_mem_usage=True,

            use_safetensors=True
        )

        model.eval()

        # ----------------------------------------------------
        # System prompt
        # ----------------------------------------------------

        system_prompt = """
You are an expert short-form technology video scriptwriter.

Generate a structured JSON video plan.

STRICT OUTPUT REQUIREMENTS:

1. Output ONLY valid JSON.
2. Generate exactly 10 scenes.
3. Scene numbers must be 1 through 10.
4. Every scene must have exactly 5.0 seconds duration.
5. Total video duration must be exactly 50 seconds.
6. Each voiceover should contain approximately 11-14 words.
7. Each subtitle should contain approximately 4-7 words.
8. Visual prompts must describe cinematic visuals in vertical 9:16 format.
9. Visual prompts must not request readable text, captions or logos.

SOURCE-GROUNDING REQUIREMENTS:

1. Use only information supported by the supplied research brief.
2. Never invent statistics, percentages, dates, quotations or performance claims.
3. Never invent company announcements or research findings.
4. Do not describe educational fallback material as breaking news.
5. Do not claim a source has been independently verified.
6. Separate documented information from interpretation.
7. If information is unavailable, acknowledge the limitation.
8. Do not invent practical deployments or real-world achievements.

The narrative must have:
- An engaging opening.
- A simple explanation of the topic.
- Available source information.
- Technical context.
- Limitations or unanswered questions.
- A concise conclusion and call to action.

REQUIRED JSON SCHEMA:

{
  "selected_topic": "string",
  "hook_title": "string",
  "target_audience": "string",
  "scenes": [
    {
      "scene_number": 1,
      "visual_prompt": "string",
      "voiceover_text": "string",
      "subtitle_text": "string",
      "duration_seconds": 5.0
    }
  ],
  "social_caption": "string",
  "hashtags": [
    "#Technology",
    "#AI",
    "#Innovation",
    "#Engineering"
  ]
}
"""

        user_message = (
            "Create a 50-second technology video using "
            "the following research brief.\n\n"
            "RESEARCH BRIEF:\n"
            + json.dumps(
                brief,
                ensure_ascii=False,
                indent=2
            )
            + "\n\n"
            "Preserve evidence limitations. "
            "Do not add unsupported claims."
        )

        messages = [

            {
                "role": "system",
                "content": system_prompt
            },

            {
                "role": "user",
                "content": user_message
            }
        ]

        prompt_text = tokenizer.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True
        )

        inputs = tokenizer(

            prompt_text,

            return_tensors="pt",

            truncation=True,

            max_length=4096

        ).to("cuda")

        print("Generating 10-scene video script...")

        with torch.inference_mode():

            outputs = model.generate(

                **inputs,

                max_new_tokens=1800,

                do_sample=True,

                temperature=0.4,

                top_p=0.85,

                repetition_penalty=1.08,

                pad_token_id=tokenizer.eos_token_id,

                eos_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[
            0,
            inputs.input_ids.shape[1]:
        ]

        output_text = tokenizer.decode(

            generated_tokens,

            skip_special_tokens=True
        )

        parsed_json = extract_json_from_response(output_text)

        plan = VideoPlan.model_validate(parsed_json)

        print(
            "\nQwen successfully generated a valid 10-scene plan."
        )

    except Exception as exc:

        print(f"\nQwen generation failed: {exc}")

        plan = None

    finally:

        # ----------------------------------------------------
        # Explicitly release every large object
        # ----------------------------------------------------

        inputs = None
        outputs = None
        generated_tokens = None

        model = None
        tokenizer = None

        prompt_text = None
        output_text = None
        parsed_json = None

        release_llm_memory()

        print("Qwen memory cleanup completed.")

    # --------------------------------------------------------
    # Return generated plan or fallback
    # --------------------------------------------------------

    if plan is not None:

        print("Using the Qwen-generated VideoPlan.")

        return plan

    print("Using grounded fallback script.")

    return generate_fallback_script(brief)


# ============================================================
# 6. CELL INITIALIZATION
# ============================================================

print("\nAI SCRIPT GENERATION ENGINE READY")
print("=" * 55)

print("Model: Qwen2.5-7B-Instruct")
print("Quantization: 4-bit NF4")
print("Target GPU: CUDA / T4")
print("Video scenes: 10")
print("Scene duration: 5 seconds")
print("Total duration: 50 seconds")
print(f"Minimum RAM for Qwen: {MIN_RAM_FOR_QWEN_GB} GB")

print("=" * 55)

# IMPORTANT:
# Do not execute Qwen here.
# Cell 12 will call generate_video_script_llm() when needed.



AI SCRIPT GENERATION ENGINE READY
Model: Qwen2.5-7B-Instruct
Quantization: 4-bit NF4
Target GPU: CUDA / T4
Video scenes: 10
Scene duration: 5 seconds
Total duration: 50 seconds
Minimum RAM for Qwen: 6.5 GB


## CELL 7: CPU Text-to-Speech Engine (Piper TTS)
- **Zero GPU Memory:** Executes 100% on CPU
- **Model:** `Piper TTS` using voice `en_US-lessac-medium` (~63 MB ONNX model)
- Auto-downloads model files to Google Drive (`/content/drive/MyDrive/VideoFactory/models/piper/`)
- Normalizes volume to -14 LUFS and trims boundary silence using FFmpeg
- Fallback: `espeak-ng` offline speech synthesis
- **Never generates fake noise as narration**


In [ ]:

# CELL 7: CPU Text-to-Speech Engine (Piper TTS)

import os
import sys
import shutil
import wave
import subprocess
import urllib.request
import gc

from pathlib import Path
from typing import Tuple, Optional

try:
    import soundfile as sf
except ImportError:
    sf = None


PIPER_VOICE_NAME = "en_US-lessac-medium"

PIPER_MODEL_URL = (
    f"https://huggingface.co/rhasspy/piper-voices/resolve/main/"
    f"en/en_US/lessac/medium/{PIPER_VOICE_NAME}.onnx"
)

PIPER_CONFIG_URL = (
    f"https://huggingface.co/rhasspy/piper-voices/resolve/main/"
    f"en/en_US/lessac/medium/{PIPER_VOICE_NAME}.onnx.json"
)

_CACHED_PIPER_VOICE = None
_LAST_TTS_ENGINE = None


def download_verified_file(url: str, destination: Path, min_size: int):
    """Download a file to a temporary path and validate its size."""

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size >= min_size:
        return

    temp_path = destination.with_suffix(destination.suffix + ".part")

    try:
        if temp_path.exists():
            temp_path.unlink()

        request = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        with urllib.request.urlopen(request, timeout=120) as response:
            if response.status != 200:
                raise RuntimeError(
                    f"Download failed with HTTP status {response.status}"
                )

            with open(temp_path, "wb") as file:
                shutil.copyfileobj(response, file)

        if not temp_path.exists() or temp_path.stat().st_size < min_size:
            raise RuntimeError(f"Downloaded file is incomplete: {destination.name}")

        temp_path.replace(destination)

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


def ensure_piper_voice_model(model_dir: Path) -> Tuple[str, str]:

    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)

    onnx_path = model_dir / f"{PIPER_VOICE_NAME}.onnx"
    json_path = model_dir / f"{PIPER_VOICE_NAME}.onnx.json"

    if not json_path.exists() or json_path.stat().st_size < 100:
        print("Downloading Piper voice configuration...")
        download_verified_file(PIPER_CONFIG_URL, json_path, 100)

    if not onnx_path.exists() or onnx_path.stat().st_size < 1_000_000:
        print("Downloading Piper voice model (~63 MB)...")
        download_verified_file(PIPER_MODEL_URL, onnx_path, 1_000_000)

    return str(onnx_path), str(json_path)


def normalize_and_trim_audio(
    raw_wav: str,
    output_wav: str,
    speed: float = 1.0
):
    """Normalize audio, remove excess silence and apply playback speed."""

    if not 0.5 <= speed <= 2.0:
        raise ValueError("Audio speed must be between 0.5 and 2.0")

    filter_chain = (
        f"atempo={speed},"
        "silenceremove=start_periods=1:start_duration=0.04:"
        "start_threshold=-45dB:"
        "stop_periods=1:stop_duration=0.08:"
        "stop_threshold=-45dB,"
        "loudnorm=I=-16:TP=-1.5:LRA=11"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", str(raw_wav),
        "-af", filter_chain,
        "-ac", "1",
        "-ar", "22050",
        "-c:a", "pcm_s16le",
        str(output_wav)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=120
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"FFmpeg audio processing failed: {result.stderr[-1000:]}"
        )

    output_path = Path(output_wav)

    if not output_path.exists() or output_path.stat().st_size < 1000:
        raise RuntimeError("Normalized audio file is missing or too small")

    with wave.open(str(output_path), "rb") as wav:
        if wav.getnframes() <= 0 or wav.getframerate() <= 0:
            raise RuntimeError("Normalized WAV contains invalid audio data")


def get_audio_duration(file_path: str) -> float:
    """Return audio duration. Raise an error if it cannot be determined."""

    if sf is not None:
        try:
            info = sf.info(file_path)
            duration = float(info.duration)

            if duration > 0:
                return duration

        except Exception:
            pass

    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(file_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=True,
        timeout=30
    )

    try:
        duration = float(result.stdout.strip())
    except (ValueError, TypeError):
        raise RuntimeError(f"Could not determine audio duration: {file_path}")

    if duration <= 0:
        raise RuntimeError(f"Invalid audio duration: {file_path}")

    return duration


def synthesize_scene_voice(
    text: str,
    output_wav: str,
    model_dir: Optional[Path] = None,
    speed: float = 1.05
) -> str:

    global _LAST_TTS_ENGINE

    text = str(text).strip()

    if not text:
        raise ValueError("Cannot synthesize empty narration")

    if not 0.5 <= speed <= 2.0:
        raise ValueError("Speed must be between 0.5 and 2.0")

    out_path = Path(output_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    temp_wav = str(out_path.with_suffix(".tmp.wav"))

    _LAST_TTS_ENGINE = None

    if model_dir is None:
        model_dir = DIRS["models_piper"]

    if Path(temp_wav).exists():
        Path(temp_wav).unlink()

    if out_path.exists():
        out_path.unlink()

    # 1. Piper TTS
    try:
        onnx_file, json_file = ensure_piper_voice_model(model_dir)

        piper_executable = shutil.which("piper")

        if piper_executable:
            cmd = [
                piper_executable,
                "-m", onnx_file,
                "-c", json_file,
                "-f", temp_wav
            ]
        else:
            cmd = [
                sys.executable,
                "-m", "piper",
                "-m", onnx_file,
                "-c", json_file,
                "-f", temp_wav
            ]

        result = subprocess.run(
            cmd,
            input=text,
            text=True,
            capture_output=True,
            timeout=120
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Piper CLI failed: {result.stderr[-1000:]}"
            )

        if not Path(temp_wav).exists() or Path(temp_wav).stat().st_size < 1000:
            raise RuntimeError("Piper did not produce a valid WAV file")

        with wave.open(temp_wav, "rb") as wav:
            if wav.getnframes() <= 0 or wav.getframerate() <= 0:
                raise RuntimeError("Piper produced invalid WAV metadata")

        normalize_and_trim_audio(
            temp_wav,
            str(out_path),
            speed=speed
        )

        duration = get_audio_duration(str(out_path))

        if duration <= 0:
            raise RuntimeError("Piper produced empty audio")

        _LAST_TTS_ENGINE = "Piper"

        logger.info(
            "Piper synthesis successful: %s (%.2f seconds)",
            out_path,
            duration
        )

        return str(out_path)

    except Exception as piper_error:
        logger.warning("Piper synthesis failed: %s", piper_error)

    finally:
        if Path(temp_wav).exists():
            Path(temp_wav).unlink()

    # 2. espeak-ng fallback
    try:
        cmd_espeak = [
            "espeak-ng",
            "-v", "en-us",
            "-s", "160",
            "-w", temp_wav,
            text
        ]

        subprocess.run(
            cmd_espeak,
            capture_output=True,
            text=True,
            check=True,
            timeout=120
        )

        if not Path(temp_wav).exists() or Path(temp_wav).stat().st_size < 500:
            raise RuntimeError("espeak-ng did not produce valid audio")

        normalize_and_trim_audio(
            temp_wav,
            str(out_path),
            speed=speed
        )

        duration = get_audio_duration(str(out_path))

        if duration <= 0:
            raise RuntimeError("espeak-ng produced empty audio")

        _LAST_TTS_ENGINE = "espeak-ng"

        logger.warning(
            "Narration generated using espeak-ng fallback: %s",
            out_path
        )

        return str(out_path)

    except Exception as fallback_error:
        logger.error("espeak-ng fallback failed: %s", fallback_error)

        if Path(out_path).exists():
            Path(out_path).unlink()

        raise RuntimeError(
            f"Both Piper and espeak-ng failed: {fallback_error}"
        ) from fallback_error

    finally:
        if Path(temp_wav).exists():
            Path(temp_wav).unlink()


## CELL 8: Visual Generation Engine (SDXL-Lightning + Ken Burns, 512×896 T4-Friendly)
- **Primary:** `ByteDance/SDXL-Lightning` 4-step UNet at T4-friendly **512×896** (9:16 vertical resolution)
- Generates high-quality vertical visuals in ~2–3 seconds per scene with ~5–6 GB VRAM footprint
- Converts each image into a 5-second 720×1280 25fps video clip using 10 alternating cinematic motion profiles
- **Resumable:** Skips completed valid scene clips (>50 KB)
- Explicitly evicts diffusion model and flushes CUDA cache upon completion


In [ ]:

# CELL 8: MEMORY-OPTIMIZED HYBRID VISUAL GENERATION ENGINE
#
# Architecture:
# 1. Wan 2.1 T2V: Selected scenes, when RAM permits
# 2. SDXL-Lightning: Keyframes, when RAM permits
# 3. FFmpeg: Animated motion graphics, no zoom effects
# 4. Procedural visuals: Safe fallback
#
# Optimized for Google Colab T4.
# No paid APIs or services.

import os
import gc
import json
import math
import hashlib
import subprocess

from pathlib import Path
from typing import Any, List

from PIL import Image, ImageDraw


# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

VIDEO_WIDTH = 720
VIDEO_HEIGHT = 1280
VIDEO_FPS = 25

SDXL_WIDTH = 512
SDXL_HEIGHT = 896

WAN_SCENE_NUMBERS = {1, 6}
WAN_NUM_FRAMES = 33
WAN_FPS = 16

SDXL_STEPS = 4
SDXL_GUIDANCE = 0.0

ENABLE_MOTION_GRAPHICS = True

CACHE_VERSION = "hybrid_visual_v3"

# Conservative host RAM thresholds.
# Wan is more memory-intensive than the procedural fallback.
WAN_MIN_RAM_GB = 9.5
SDXL_MIN_RAM_GB = 8.0

# Do not load models if the runtime is already under pressure.
MIN_RAM_SAFETY_GB = 2.0


# ---------------------------------------------------------
# 2. SYSTEM AND GPU MEMORY MANAGEMENT
# ---------------------------------------------------------

def get_available_system_ram_gb():

    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")

        return (pages * page_size) / (1024 ** 3)

    except Exception:
        return 0.0


def get_visual_gpu_memory():

    try:
        import torch

        if not torch.cuda.is_available():
            return 0.0

        free_bytes, total_bytes = torch.cuda.mem_get_info(0)

        return free_bytes / (1024 ** 3)

    except Exception:
        return 0.0


def clean_visual_gpu_memory(label="Visual Engine"):

    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    except Exception as exc:
        print(f"GPU cleanup warning: {exc}")

    gc.collect()

    print(f"Memory cleanup completed: {label}")


def print_visual_memory():

    ram = get_available_system_ram_gb()
    vram = get_visual_gpu_memory()

    print(
        f"Available system RAM: {ram:.2f} GB | "
        f"Available GPU VRAM: {vram:.2f} GB"
    )

    return ram, vram


# ---------------------------------------------------------
# 3. GENERAL HELPERS
# ---------------------------------------------------------

def visual_get_field(obj, names, default=None):

    if isinstance(obj, dict):
        data = obj
    elif hasattr(obj, "model_dump"):
        data = obj.model_dump()
    else:
        data = vars(obj) if hasattr(obj, "__dict__") else {}

    for name in names:
        value = data.get(name)

        if value is not None:
            return value

    for name in names:
        if hasattr(obj, name):
            value = getattr(obj, name)

            if value is not None:
                return value

    return default


def visual_topic(video_plan):

    return str(
        visual_get_field(
            video_plan,
            ["topic", "selected_topic", "topic_title", "title"],
            "Artificial Intelligence"
        )
    )


def get_visual_signature(data):

    serialized = json.dumps(
        data,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def get_scene_signature(prompt, duration, engine):

    return get_visual_signature({
        "prompt": str(prompt),
        "duration": round(float(duration), 3),
        "engine": str(engine),
        "width": VIDEO_WIDTH,
        "height": VIDEO_HEIGHT,
        "fps": VIDEO_FPS,
        "version": CACHE_VERSION
    })


def save_visual_signature(path, signature):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.stem + "_temp.json")

    try:
        temporary.write_text(
            json.dumps({"signature": signature}, indent=2),
            encoding="utf-8"
        )

        temporary.replace(path)

    finally:
        if temporary.exists():
            temporary.unlink()


def signature_matches(path, expected):

    path = Path(path)

    if not path.exists():
        return False

    try:
        data = json.loads(path.read_text(encoding="utf-8"))

        return data.get("signature") == expected

    except Exception:
        return False


# ---------------------------------------------------------
# 4. FILE VALIDATION
# ---------------------------------------------------------

def validate_video_file(video_path):

    path = Path(video_path)

    if not path.exists() or path.stat().st_size < 10000:
        return False

    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height",
        "-of", "json",
        str(path)
    ]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True,
            timeout=30
        )

        streams = json.loads(result.stdout).get("streams", [])

        return bool(
            streams
            and streams[0].get("width", 0) > 0
            and streams[0].get("height", 0) > 0
        )

    except Exception:
        return False


def validate_image_file(image_path):

    path = Path(image_path)

    if not path.exists() or path.stat().st_size < 1000:
        return False

    try:
        with Image.open(path) as image:
            image.verify()

        return True

    except Exception:
        return False


def image_is_reusable(image_path, signature_path, signature):

    return (
        validate_image_file(image_path)
        and signature_matches(signature_path, signature)
    )


def scene_clip_is_reusable(clip_path, signature_path, signature):

    return (
        validate_video_file(clip_path)
        and signature_matches(signature_path, signature)
    )


# ---------------------------------------------------------
# 5. MOTION GRAPHICS RENDERER
# ---------------------------------------------------------

def render_image_to_motion_video(
    image_path,
    output_mp4,
    duration=5.0,
    scene_index=0,
    fps=VIDEO_FPS
):

    image_path = Path(image_path)
    output_path = Path(output_mp4)

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    if duration <= 0:
        raise ValueError("Scene duration must be positive")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    total_frames = max(1, int(round(duration * fps)))

    temp_path = output_path.with_name(
        output_path.stem + ".tmp.mp4"
    )

    if temp_path.exists():
        temp_path.unlink()

    vf = (
        f"scale={VIDEO_WIDTH}:{VIDEO_HEIGHT}:"
        "force_original_aspect_ratio=increase,"
        f"crop={VIDEO_WIDTH}:{VIDEO_HEIGHT},"
        "setsar=1,"
        "format=yuv420p"
    )

    if ENABLE_MOTION_GRAPHICS:

        vf += (
            ",drawbox=x='mod(t*180\\,900)-120':"
            "y=0:w=100:h=ih:"
            "color=white@0.065:t=fill"
        )

        vf += (
            ",drawbox=x='mod(t*260\\,900)-40':"
            "y=ih*0.84:w=100:h=4:"
            "color=white@0.65:t=fill"
        )

        vf += (
            ",drawbox=x=iw*0.045:"
            "y='mod(t*100\\,ih)':"
            "w=3:h=100:"
            "color=white@0.35:t=fill"
        )

    cmd = [
        "ffmpeg",
        "-y",
        "-loop", "1",
        "-i", str(image_path),
        "-vf", vf,
        "-frames:v", str(total_frames),
        "-an",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-crf", "21",
        "-pix_fmt", "yuv420p",
        "-r", str(fps),
        "-movflags", "+faststart",
        str(temp_path)
    ]

    try:

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=max(120, int(duration * 30))
        )

        if result.returncode != 0:
            raise RuntimeError(result.stderr[-2000:])

        if not validate_video_file(temp_path):
            raise RuntimeError("Motion video failed validation")

        temp_path.replace(output_path)

        return str(output_path)

    finally:

        if temp_path.exists():
            temp_path.unlink()


# Compatibility alias. No zoom effects.
def render_image_to_kenburns_video(
    image_path,
    output_mp4,
    duration=5.0,
    scene_index=0,
    fps=VIDEO_FPS
):

    return render_image_to_motion_video(
        image_path,
        output_mp4,
        duration,
        scene_index,
        fps
    )


# ---------------------------------------------------------
# 6. PROCEDURAL VISUAL FALLBACK
# ---------------------------------------------------------

def generate_procedural_scene_image(
    prompt,
    output_image_path,
    scene_number,
    topic
):

    width = 512
    height = 896

    image = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(image)

    palettes = [
        ((15, 23, 42), (59, 130, 246)),
        ((10, 10, 20), (168, 85, 247)),
        ((6, 78, 59), (16, 185, 129)),
        ((30, 27, 75), (236, 72, 153)),
        ((24, 24, 27), (245, 158, 11))
    ]

    start, end = palettes[(scene_number - 1) % len(palettes)]

    for y in range(height):

        t = y / max(1, height - 1)

        color = tuple(
            int(start[i] * (1 - t) + end[i] * t)
            for i in range(3)
        )

        draw.line([(0, y), (width, y)], fill=color)

    for i in range(8):

        offset = i * 90

        draw.ellipse(
            [
                50 + i * 12,
                140 + offset,
                width - 50 - i * 12,
                280 + offset
            ],
            outline=(255, 255, 255),
            width=2
        )

    output_path = Path(output_image_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image.save(output_path, format="PNG")

    return str(output_path)


# ---------------------------------------------------------
# 7. SDXL-LIGHTNING PIPELINE
# ---------------------------------------------------------

def load_sdxl_lightning_pipeline():

    import torch

    from diffusers import (
        StableDiffusionXLPipeline,
        EulerDiscreteScheduler
    )

    if not torch.cuda.is_available():
        raise RuntimeError("SDXL requires CUDA")

    print("Loading SDXL-Lightning in memory-efficient mode...")

    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
        low_cpu_mem_usage=True
    )

    pipe.load_lora_weights(
        "ByteDance/SDXL-Lightning",
        weight_name="sdxl_lightning_4step_lora.safetensors"
    )

    pipe.fuse_lora()

    pipe.scheduler = EulerDiscreteScheduler.from_config(
        pipe.scheduler.config,
        timestep_spacing="trailing"
    )

    # CPU offload prevents the entire pipeline staying on GPU.
    pipe.enable_model_cpu_offload()

    pipe.enable_attention_slicing()

    if hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()

    if hasattr(pipe, "enable_vae_tiling"):
        pipe.enable_vae_tiling()

    return pipe


def generate_sdxl_keyframes(
    scenes,
    image_dir,
    topic
):

    import torch

    image_dir = Path(image_dir)
    image_dir.mkdir(parents=True, exist_ok=True)

    # Check cache BEFORE loading the model.
    pending = []

    for scene in scenes:

        number = int(scene.scene_number)

        image_path = image_dir / f"scene_{number:02d}.png"
        signature_path = image_dir / f"scene_{number:02d}_image.json"

        prompt = str(scene.visual_prompt)

        signature = get_visual_signature({
            "prompt": prompt,
            "engine": "sdxl_lightning",
            "version": CACHE_VERSION,
            "width": SDXL_WIDTH,
            "height": SDXL_HEIGHT
        })

        if image_is_reusable(
            image_path,
            signature_path,
            signature
        ):

            print(f"Scene {number}: Cached SDXL image reused.")

        else:

            pending.append(
                (scene, image_path, signature_path, signature)
            )

    if not pending:
        print("All SDXL keyframes are already cached.")
        return

    ram = get_available_system_ram_gb()

    if ram < SDXL_MIN_RAM_GB:
        print(
            f"SDXL skipped: only {ram:.2f} GB system RAM available."
        )
        return

    pipe = None

    try:

        pipe = load_sdxl_lightning_pipeline()

        for scene, image_path, signature_path, signature in pending:

            number = int(scene.scene_number)

            # Recheck memory between scenes.
            ram = get_available_system_ram_gb()

            if ram < MIN_RAM_SAFETY_GB:
                print(
                    "SDXL stopped: system RAM reached safety threshold."
                )
                break

            prompt = (
                f"{scene.visual_prompt}, "
                "cinematic editorial photography, "
                "realistic lighting, detailed environment, "
                "professional composition, vertical 9:16"
            )

            negative_prompt = (
                "text, watermark, blurry, distorted, "
                "low quality, deformed, oversaturated"
            )

            print(
                f"Generating SDXL keyframe for scene {number}..."
            )

            with torch.inference_mode():

                result = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=SDXL_STEPS,
                    guidance_scale=SDXL_GUIDANCE,
                    width=SDXL_WIDTH,
                    height=SDXL_HEIGHT
                )

                generated_image = result.images[0]

            temp_path = image_path.with_name(
                image_path.stem + "_temp.png"
            )

            generated_image.save(temp_path, format="PNG")

            temp_path.replace(image_path)

            save_visual_signature(
                signature_path,
                signature
            )

            del generated_image
            del result

            clean_visual_gpu_memory(
                f"SDXL scene {number}"
            )

    finally:

        if pipe is not None:
            del pipe

        clean_visual_gpu_memory("SDXL pipeline released")


# ---------------------------------------------------------
# 8. WAN 2.1 PIPELINE
# ---------------------------------------------------------

def load_wan_pipeline():

    import torch

    from diffusers import WanPipeline

    if not torch.cuda.is_available():
        raise RuntimeError("Wan requires CUDA")

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    print("Loading Wan 2.1 T2V 1.3B...")

    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    )

    pipe.enable_model_cpu_offload()

    if hasattr(pipe, "enable_vae_tiling"):
        pipe.enable_vae_tiling()

    return pipe


def normalize_wan_video(input_video, output_video, duration):

    output_path = Path(output_video)

    temp_path = output_path.with_name(
        output_path.stem + ".tmp.mp4"
    )

    if temp_path.exists():
        temp_path.unlink()

    cmd = [
        "ffmpeg",
        "-y",
        "-stream_loop", "-1",
        "-i", str(input_video),
        "-vf",
        (
            "scale=720:1280:"
            "force_original_aspect_ratio=increase,"
            "crop=720:1280,"
            "fps=25,"
            "setsar=1"
        ),
        "-t", f"{duration:.3f}",
        "-an",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-crf", "21",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(temp_path)
    ]

    try:

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=max(120, int(duration * 40))
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Wan normalization failed: {result.stderr[-1500:]}"
            )

        if not validate_video_file(temp_path):
            raise RuntimeError("Normalized Wan video is invalid")

        temp_path.replace(output_path)

    finally:

        if temp_path.exists():
            temp_path.unlink()


def generate_wan_video_assets(
    scenes,
    scene_dir,
    durations,
    required_scenes
):

    import torch
    from diffusers.utils import export_to_video

    scene_dir = Path(scene_dir)

    if not required_scenes:
        print("No Wan scenes need generation.")
        return set()

    # Check memory before loading Wan.
    ram = get_available_system_ram_gb()

    if ram < WAN_MIN_RAM_GB:
        print(
            f"Wan skipped: {ram:.2f} GB RAM available; "
            f"{WAN_MIN_RAM_GB:.1f} GB required by safety policy."
        )
        return set()

    pipe = None
    generated = set()

    try:

        pipe = load_wan_pipeline()

        for index, scene in enumerate(scenes):

            number = int(scene.scene_number)

            if number not in required_scenes:
                continue

            # Stop before memory exhaustion.
            ram = get_available_system_ram_gb()

            if ram < MIN_RAM_SAFETY_GB:
                print("Wan stopped due to low available system RAM.")
                break

            duration = float(durations[index])

            clip_path = scene_dir / f"scene_{number:02d}.mp4"
            signature_path = scene_dir / f"scene_{number:02d}.json"

            raw_path = scene_dir / f"scene_{number:02d}_wan_raw.mp4"

            signature = get_scene_signature(
                scene.visual_prompt,
                duration,
                "wan"
            )

            prompt = (
                f"{scene.visual_prompt}, "
                "cinematic video, natural physical movement, "
                "realistic motion, detailed environment, "
                "vertical composition"
            )

            print(f"Generating Wan video for scene {number}...")

            with torch.inference_mode():

                output = pipe(
                    prompt=prompt,
                    height=832,
                    width=480,
                    num_frames=WAN_NUM_FRAMES,
                    guidance_scale=5.0
                )

            export_to_video(
                output.frames[0],
                str(raw_path),
                fps=WAN_FPS
            )

            del output

            if not validate_video_file(raw_path):
                raise RuntimeError(
                    f"Wan scene {number} failed validation"
                )

            normalize_wan_video(
                str(raw_path),
                str(clip_path),
                duration
            )

            save_visual_signature(
                signature_path,
                signature
            )

            if raw_path.exists():
                raw_path.unlink()

            generated.add(number)

            clean_visual_gpu_memory(
                f"After Wan scene {number}"
            )

    finally:

        if pipe is not None:
            del pipe

        clean_visual_gpu_memory("Wan pipeline released")

    return generated


# ---------------------------------------------------------
# 9. MAIN HYBRID VISUAL GENERATION ORCHESTRATOR
# ---------------------------------------------------------

def generate_visual_assets(
    video_plan: Any,
    image_dir: Path,
    scene_dir: Path,
    durations: List[float],
    engine: str = "hybrid"
):

    image_dir = Path(image_dir)
    scene_dir = Path(scene_dir)

    image_dir.mkdir(parents=True, exist_ok=True)
    scene_dir.mkdir(parents=True, exist_ok=True)

    scenes = video_plan.scenes

    if len(scenes) != len(durations):
        raise ValueError(
            "Number of scenes and durations do not match"
        )

    if any(
        not math.isfinite(float(d)) or float(d) <= 0
        for d in durations
    ):
        raise ValueError("Invalid scene duration")

    print("\n" + "=" * 60)
    print("MEMORY-OPTIMIZED HYBRID VISUAL GENERATION")
    print("=" * 60)

    ram, vram = print_visual_memory()

    engine = engine.lower().strip()

    if engine not in ("hybrid", "wan", "sdxl", "procedural"):
        raise ValueError(f"Unsupported visual engine: {engine}")

    topic = visual_topic(video_plan)

    # -----------------------------------------------------
    # A. CHECK EXISTING VIDEO CACHE
    # -----------------------------------------------------

    reusable = set()
    pending = []

    for index, scene in enumerate(scenes):

        number = int(scene.scene_number)
        duration = float(durations[index])

        clip_path = scene_dir / f"scene_{number:02d}.mp4"
        signature_path = scene_dir / f"scene_{number:02d}.json"

        if engine in ("hybrid", "wan") and number in WAN_SCENE_NUMBERS:
            expected_engine = "wan"
        elif engine == "procedural":
            expected_engine = "procedural"
        else:
            expected_engine = "sdxl_motion"

        signature = get_scene_signature(
            scene.visual_prompt,
            duration,
            expected_engine
        )

        if scene_clip_is_reusable(
            clip_path,
            signature_path,
            signature
        ):

            print(f"Scene {number}: Existing video reused.")
            reusable.add(number)

        else:
            pending.append(index)

    if not pending:
        print("All scene videos are already cached.")
        return

    # -----------------------------------------------------
    # B. WAN VIDEO GENERATION
    # -----------------------------------------------------

    wan_generated = set()

    if engine in ("hybrid", "wan"):

        required_wan = {
            int(scenes[i].scene_number)
            for i in pending
            if int(scenes[i].scene_number) in WAN_SCENE_NUMBERS
        }

        if required_wan:

            try:

                wan_generated = generate_wan_video_assets(
                    scenes,
                    scene_dir,
                    durations,
                    required_wan
                )

            except Exception as exc:

                print(f"Wan unavailable; using fallback: {exc}")

                clean_visual_gpu_memory("Wan recovery")

    # -----------------------------------------------------
    # C. SDXL KEYFRAME GENERATION
    # -----------------------------------------------------

    remaining = [
        scenes[i]
        for i in pending
        if int(scenes[i].scene_number) not in wan_generated
    ]

    if engine in ("hybrid", "sdxl") and remaining:

        try:

            generate_sdxl_keyframes(
                remaining,
                image_dir,
                topic
            )

        except Exception as exc:

            print(f"SDXL unavailable; using fallback: {exc}")

            clean_visual_gpu_memory("SDXL recovery")

    # -----------------------------------------------------
    # D. FINAL SCENE RENDERING
    # -----------------------------------------------------

    for index, scene in enumerate(scenes):

        number = int(scene.scene_number)
        duration = float(durations[index])

        if number in reusable or number in wan_generated:
            continue

        clip_path = scene_dir / f"scene_{number:02d}.mp4"
        clip_signature_path = scene_dir / f"scene_{number:02d}.json"

        image_path = image_dir / f"scene_{number:02d}.png"
        image_signature_path = (
            image_dir / f"scene_{number:02d}_image.json"
        )

        # -------------------------------------------------
        # Check SDXL image cache.
        # -------------------------------------------------

        image_signature = get_visual_signature({
            "prompt": str(scene.visual_prompt),
            "engine": "sdxl_lightning",
            "version": CACHE_VERSION,
            "width": SDXL_WIDTH,
            "height": SDXL_HEIGHT
        })

        image_ready = image_is_reusable(
            image_path,
            image_signature_path,
            image_signature
        )

        # -------------------------------------------------
        # If no SDXL image exists, create procedural image.
        # -------------------------------------------------

        if not image_ready:

            print(
                f"Scene {number}: Creating procedural fallback image."
            )

            generate_procedural_scene_image(
                scene.visual_prompt,
                str(image_path),
                number,
                topic
            )

            procedural_signature = get_visual_signature({
                "prompt": str(scene.visual_prompt),
                "engine": "procedural",
                "version": CACHE_VERSION
            })

            save_visual_signature(
                image_signature_path,
                procedural_signature
            )

        # -------------------------------------------------
        # Render animated scene video.
        # -------------------------------------------------

        print(f"Scene {number}: Rendering motion graphics.")

        render_image_to_motion_video(
            str(image_path),
            str(clip_path),
            duration,
            index
        )

        if not validate_video_file(clip_path):
            raise RuntimeError(
                f"Scene {number} failed final validation"
            )

        # Use the correct signature for the actual image source.
        if image_ready:
            render_engine = "sdxl_motion"
        else:
            render_engine = "procedural"

        save_visual_signature(
            clip_signature_path,
            get_scene_signature(
                scene.visual_prompt,
                duration,
                render_engine
            )
        )

        clean_visual_gpu_memory(
            f"Scene {number} completed"
        )

    # -----------------------------------------------------
    # E. FINAL ASSET VALIDATION
    # -----------------------------------------------------

    missing = []

    for scene in scenes:

        number = int(scene.scene_number)

        clip_path = scene_dir / f"scene_{number:02d}.mp4"

        if not validate_video_file(clip_path):
            missing.append(number)

    if missing:
        raise RuntimeError(
            f"Visual generation incomplete. Invalid scenes: {missing}"
        )

    print("\n" + "=" * 60)
    print("HYBRID VISUAL GENERATION COMPLETED")
    print(f"Total scenes: {len(scenes)}")
    print(f"Wan AI scenes: {len(wan_generated)}")
    print(
        f"Image + motion scenes: "
        f"{len(scenes) - len(wan_generated)}"
    )

    print_visual_memory()

    print("=" * 60)


print("Memory-Optimized Hybrid Visual Engine ready.")
print("Wan AI video + SDXL keyframes + procedural fallback.")
print("Prompt-aware caching enabled.")
print("Ken Burns zoom effects: DISABLED.")


Memory-Optimized Hybrid Visual Engine ready.
Wan AI video + SDXL keyframes + procedural fallback.
Prompt-aware caching enabled.
Ken Burns zoom effects: DISABLED.


## CELL 9: Subtitle Burning and Audio-Video Assembly
- Styles animated subtitles: centered horizontally, 76% height, bold white font with black outline
- Burns subtitles directly into video frames with zero timing drift
- Handles file paths containing spaces safely
- Concatenates all 10 synchronized scenes using FFmpeg concat demuxer with re-encoding safety


In [ ]:

# CELL 9: Subtitles & Video Editing Engine

import subprocess
import math
from pathlib import Path


def format_ass_time(seconds: float) -> str:
    """Convert seconds to valid ASS timestamp: H:MM:SS.cc."""
    if not math.isfinite(seconds) or seconds < 0:
        raise ValueError(f"Invalid timestamp: {seconds}")

    total_cs = round(seconds * 100)
    hrs, remainder = divmod(total_cs, 360000)
    mins, remainder = divmod(remainder, 6000)
    secs, cs = divmod(remainder, 100)

    return f"{hrs}:{mins:02d}:{secs:02d}.{cs:02d}"


def wrap_subtitle_text(text: str, max_words_per_line: int = 5) -> str:
    """Wrap subtitle text into lines of at most max_words_per_line words."""
    if max_words_per_line < 1:
        raise ValueError("max_words_per_line must be at least 1")

    words = text.strip().split()
    if not words:
        return ""

    lines = [
        " ".join(words[i:i + max_words_per_line]).upper()
        for i in range(0, len(words), max_words_per_line)
    ]

    return r"\N".join(lines)


def create_ass_file_for_clip(
    subtitle_text: str,
    duration: float,
    output_ass: str,
    font_name: str = "DejaVuSans-Bold"
) -> str:
    """Create an ASS subtitle file for a vertical video."""
    if duration <= 0.3:
        raise ValueError(f"Clip duration is too short: {duration}")

    formatted = wrap_subtitle_text(subtitle_text)
    if not formatted:
        raise ValueError("Subtitle text cannot be empty")

    start_t = format_ass_time(0.15)
    end_t = format_ass_time(duration - 0.15)

    # Escape characters that have special meaning in ASS dialogue text.
    formatted = (
        formatted
        .replace("\\", r"\\")
        .replace("{", r"\{")
        .replace("}", r"\}")
    )

    # Preserve intentional ASS line breaks.
    formatted = formatted.replace(r"\\N", r"\N")

    ass_content = (
        "[Script Info]\n"
        "ScriptType: v4.00+\n"
        "PlayResX: 720\n"
        "PlayResY: 1280\n"
        "ScaledBorderAndShadow: yes\n\n"
        "[V4+ Styles]\n"
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, "
        "OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, "
        "ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, "
        "Alignment, MarginL, MarginR, MarginV, Encoding\n"
        f"Style: ShortForm,{font_name},28,&H00FFFFFF,&H000000FF,"
        "&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,3.2,1.2,"
        "2,40,40,160,1\n\n"
        "[Events]\n"
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, "
        "MarginV, Effect, Text\n"
        f"Dialogue: 0,{start_t},{end_t},ShortForm,,0,0,0,,"
        f"{{\\b1}}{formatted}{{\\b0}}\n"
    )

    output_path = Path(output_ass)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(ass_content, encoding="utf-8")

    return str(output_path)


def burn_subtitles_and_audio_to_clip(
    video_clip: str,
    audio_clip: str,
    subtitle_text: str,
    output_clip: str,
    temp_dir: Path
) -> str:
    """Combine video, narration and burned-in subtitles."""
    video_path = Path(video_clip)
    audio_path = Path(audio_clip)
    output_path = Path(output_clip)

    if not video_path.is_file():
        raise FileNotFoundError(f"Video not found: {video_path}")

    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    if video_path.stat().st_size == 0 or audio_path.stat().st_size == 0:
        raise ValueError("Input video or audio file is empty")

    dur = get_audio_duration(str(audio_path))
    if not math.isfinite(dur) or dur <= 0.3:
        raise ValueError(f"Invalid narration duration: {dur}")

    temp_dir = Path(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    ass_path = temp_dir / f"sub_{output_path.stem}.ass"

    create_ass_file_for_clip(
        subtitle_text,
        dur,
        str(ass_path)
    )

    # Escape the subtitle path for FFmpeg's filter syntax.
    escaped_ass = (
        str(ass_path.resolve())
        .replace("\\", "/")
        .replace(":", r"\:")
        .replace("'", r"\'")
        .replace(",", r"\,")
        .replace("[", r"\[")
        .replace("]", r"\]")
    )

    temp_output = output_path.with_name(
        output_path.stem + "_temp" + output_path.suffix
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-i", str(audio_path),
        "-vf", f"subtitles='{escaped_ass}'",
        "-t", f"{dur:.3f}",
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "19",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        "-movflags", "+faststart",
        str(temp_output)
    ]

    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            raise RuntimeError(
                "Subtitle rendering failed:\n" + result.stderr[-3000:]
            )

        if (
            not temp_output.is_file()
            or temp_output.stat().st_size == 0
        ):
            raise RuntimeError("FFmpeg produced an empty output video")

        temp_output.replace(output_path)

    finally:
        if temp_output.exists():
            temp_output.unlink()

    return str(output_path)


print("✓ Subtitle & Assembly engine initialized.")


✓ Subtitle & Assembly engine initialized.


## CELL 10: Background Music & Audio Ducking
- Generates or loads royalty-free ambient background music
- Applies audio ducking: narration voice remains at 0 dB, background music ducks to -18 dB
- Adds 1.5s fade-in and 2.5s fade-out to prevent audio clipping


In [ ]:

# CELL 10: Background Music & Audio Ducking

import subprocess
import shutil
import math
from pathlib import Path
from typing import List


def run_ffmpeg_checked(cmd: List[str], operation: str) -> None:
    """Run FFmpeg and raise a useful error if the operation fails."""
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{operation} failed:\n{result.stderr[-4000:]}"
        )


def validate_media_file(path: str, description: str) -> None:
    """Check that a generated media file exists and is non-empty."""
    file_path = Path(path)

    if not file_path.is_file():
        raise FileNotFoundError(f"{description} not found: {file_path}")

    if file_path.stat().st_size == 0:
        raise RuntimeError(f"{description} is empty: {file_path}")


def generate_procedural_background_music(
    output_wav: str,
    duration: float = 60.0
) -> str:
    """
    Generate a soft brown-noise ambient background bed.

    Note: This is ambient sound, not a melodic music composition.
    """
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Background audio duration must be positive")

    output_path = Path(output_wav)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    temp_output = output_path.with_name(
        output_path.stem + "_temp.wav"
    )

    cmd = [
        "ffmpeg", "-y",
        "-f", "lavfi",
        "-i", f"anoisesrc=d={duration:.3f}:c=brown:r=44100:a=0.035",
        "-af",
        (
            "lowpass=f=450,"
            "volume=1.2,"
            "afade=t=in:st=0:d=2.0"
        ),
        "-ar", "44100",
        "-ac", "2",
        "-c:a", "pcm_s16le",
        str(temp_output)
    ]

    try:
        run_ffmpeg_checked(cmd, "Background audio generation")
        validate_media_file(str(temp_output), "Background audio")
        temp_output.replace(output_path)
    finally:
        if temp_output.exists():
            temp_output.unlink()

    return str(output_path)


def stitch_short_video(
    video_paths: List[str],
    audio_paths: List[str],
    subtitles: List[str],
    output_mp4: str,
    temp_dir: Path,
    background_music: bool = True
) -> str:
    """
    Assemble scenes, narration and subtitles into one vertical video.

    Processing:
    1. Synchronize each scene with its narration and subtitles.
    2. Normalize scene streams.
    3. Concatenate scenes.
    4. Optionally mix ambient audio with narration sidechain ducking.
    5. Validate and save the final MP4.
    """

    if not video_paths:
        raise ValueError("No scene videos were provided")

    if len(video_paths) != len(audio_paths):
        raise ValueError(
            f"Scene/audio count mismatch: "
            f"{len(video_paths)} videos, {len(audio_paths)} audio files"
        )

    if len(video_paths) != len(subtitles):
        raise ValueError(
            f"Scene/subtitle count mismatch: "
            f"{len(video_paths)} videos, {len(subtitles)} subtitles"
        )

    temp_dir = Path(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)

    output_path = Path(output_mp4)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------
    # 1. Validate inputs and synchronize each scene
    # --------------------------------------------------

    synced_scene_clips = []

    for i, (video, audio, subtitle) in enumerate(
        zip(video_paths, audio_paths, subtitles),
        start=1
    ):
        validate_media_file(video, f"Scene {i} video")
        validate_media_file(audio, f"Scene {i} narration")

        synced_out = temp_dir / f"synced_scene_{i:02d}.mp4"

        burn_subtitles_and_audio_to_clip(
            video,
            audio,
            subtitle,
            str(synced_out),
            temp_dir
        )

        validate_media_file(str(synced_out), f"Synced scene {i}")
        synced_scene_clips.append(str(synced_out))

    # --------------------------------------------------
    # 2. Normalize and concatenate all scene streams
    # --------------------------------------------------

    filter_parts = []
    concat_inputs = []

    for i in range(len(synced_scene_clips)):
        filter_parts.append(
            f"[{i}:v]"
            "scale=720:1280:force_original_aspect_ratio=decrease,"
            "pad=720:1280:(ow-iw)/2:(oh-ih)/2,"
            "setsar=1,"
            "fps=30,"
            "format=yuv420p"
            f"[v{i}]"
        )

        filter_parts.append(
            f"[{i}:a]"
            "aresample=44100,"
            "aformat=sample_fmts=fltp:channel_layouts=stereo"
            f"[a{i}]"
        )

        concat_inputs.append(f"[v{i}][a{i}]")

    filter_parts.append(
        "".join(concat_inputs)
        + f"concat=n={len(synced_scene_clips)}:v=1:a=1"
        "[vout][aout]"
    )

    filter_complex = ";".join(filter_parts)

    raw_stitched = temp_dir / "raw_stitched.mp4"
    temp_stitched = temp_dir / "raw_stitched_temp.mp4"

    cmd_concat = ["ffmpeg", "-y"]

    for clip in synced_scene_clips:
        cmd_concat.extend(["-i", clip])

    cmd_concat.extend([
        "-filter_complex", filter_complex,
        "-map", "[vout]",
        "-map", "[aout]",
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "19",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(temp_stitched)
    ])

    try:
        run_ffmpeg_checked(cmd_concat, "Scene concatenation")
        validate_media_file(str(temp_stitched), "Concatenated video")
        temp_stitched.replace(raw_stitched)
    finally:
        if temp_stitched.exists():
            temp_stitched.unlink()

    # --------------------------------------------------
    # 3. Export without background audio if disabled
    # --------------------------------------------------

    if not background_music:
        temp_output = output_path.with_name(
            output_path.stem + "_temp" + output_path.suffix
        )

        try:
            shutil.copy2(raw_stitched, temp_output)
            validate_media_file(str(temp_output), "Final video")
            temp_output.replace(output_path)
        finally:
            if temp_output.exists():
                temp_output.unlink()

        return str(output_path)

    # --------------------------------------------------
    # 4. Generate ambient audio and apply ducking
    # --------------------------------------------------

    total_dur = get_audio_duration(str(raw_stitched))

    if not math.isfinite(total_dur) or total_dur <= 0:
        raise RuntimeError("Could not determine final video duration")

    bgm_path = temp_dir / "ambient_bgm.wav"

    generate_procedural_background_music(
        str(bgm_path),
        duration=total_dur + 1.0
    )

    fade_out_start = max(0.0, total_dur - 2.5)

    filter_complex = (
        # Reduce ambient audio level and apply fades.
        "[1:a]"
        "volume=0.12,"
        "afade=t=in:st=0:d=1.5,"
        f"afade=t=out:st={fade_out_start:.3f}:d=2.5"
        "[bg];"

        # Compress background audio when narration is present.
        "[bg][0:a]"
        "sidechaincompress="
        "threshold=0.03:"
        "ratio=8:"
        "attack=20:"
        "release=500"
        "[ducked];"

        # Mix narration and ducked background audio.
        "[0:a][ducked]"
        "amix=inputs=2:duration=first:dropout_transition=2"
        "[aout]"
    )

    temp_output = output_path.with_name(
        output_path.stem + "_temp" + output_path.suffix
    )

    cmd_mix = [
        "ffmpeg", "-y",
        "-i", str(raw_stitched),
        "-i", str(bgm_path),
        "-filter_complex", filter_complex,
        "-map", "0:v",
        "-map", "[aout]",
        "-t", f"{total_dur:.3f}",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(temp_output)
    ]

    try:
        run_ffmpeg_checked(cmd_mix, "Background audio mixing")
        validate_media_file(str(temp_output), "Final video")
        temp_output.replace(output_path)
    finally:
        if temp_output.exists():
            temp_output.unlink()

    return str(output_path)


print("✓ Background Music & Audio Ducking engine initialized.")


✓ Background Music & Audio Ducking engine initialized.


## CELL 11: SEO Deliverables and Metadata Engine
- Generates platform-optimized captions for YouTube Shorts, Instagram Reels, and TikTok
- Saves `outputs/caption.txt` and `outputs/video_metadata.json`
- Logs full production telemetry specifying `Piper TTS (CPU)` as the narration engine


In [ ]:

# CELL 11: SEO Deliverables & Metadata Engine

import json
import math
from pathlib import Path
from typing import Any, Dict, List


def get_plan_value(obj: Any, *names: str, default=None):
    """Read a field using supported schema names."""
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    return default


def format_social_timestamp(seconds: float) -> str:
    """Convert elapsed seconds to M:SS."""
    total_seconds = max(0, int(seconds))
    minutes, secs = divmod(total_seconds, 60)
    return f"{minutes}:{secs:02d}"


def generate_social_deliverables(
    video_plan: Any,
    actual_durations: List[float],
    total_duration: float,
    output_dir: Path,
    engine_used: str = "hybrid"
) -> Dict[str, str]:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    scenes = video_plan.scenes

    if not scenes:
        raise ValueError("Video plan contains no scenes")

    if len(actual_durations) != len(scenes):
        raise ValueError(
            f"Scene duration mismatch: "
            f"{len(scenes)} scenes, "
            f"{len(actual_durations)} durations"
        )

    # --------------------------------------------------
    # 1. Extract canonical VideoPlan fields
    # --------------------------------------------------

    hook = str(
        get_plan_value(video_plan, "hook", "hook_title", default="")
    ).strip()

    topic = str(
        get_plan_value(video_plan, "topic", "selected_topic", default="")
    ).strip()

    audience = str(
        get_plan_value(
            video_plan,
            "audience",
            "target_audience",
            default="General technology audience"
        )
    ).strip()

    caption = str(
        get_plan_value(
            video_plan,
            "caption",
            "social_caption",
            default=""
        )
    ).strip()

    hashtags_raw = get_plan_value(
        video_plan,
        "hashtags",
        default=[]
    )

    hashtags = []

    for tag in hashtags_raw:
        tag = str(tag).strip()

        if not tag:
            continue

        if not tag.startswith("#"):
            tag = "#" + tag

        if tag not in hashtags:
            hashtags.append(tag)

    hashtags = hashtags[:6]

    if not hook:
        raise ValueError("Video plan is missing its hook")

    if not topic:
        raise ValueError("Video plan is missing its topic")

    hashtag_text = " ".join(hashtags)

    # --------------------------------------------------
    # 2. Generate actual scene telemetry and timestamps
    # --------------------------------------------------

    scene_telemetry = []
    elapsed = 0.0
    timeline_lines = []

    for index, scene in enumerate(scenes):

        duration = float(actual_durations[index])

        if not math.isfinite(duration) or duration <= 0:
            raise ValueError(
                f"Invalid duration for scene {index + 1}: {duration}"
            )

        scene_number = get_plan_value(
            scene, "scene_number", default=index + 1
        )

        visual_prompt = str(
            get_plan_value(scene, "visual_prompt", default="")
        )

        voiceover = str(
            get_plan_value(
                scene, "voiceover", "voiceover_text", default=""
            )
        )

        subtitle = str(
            get_plan_value(
                scene, "subtitle", "subtitle_text", default=""
            )
        )

        start_time = elapsed

        scene_telemetry.append({
            "scene_number": scene_number,
            "start_time_seconds": round(start_time, 3),
            "end_time_seconds": round(start_time + duration, 3),
            "duration_seconds": round(duration, 3),
            "visual_prompt": visual_prompt,
            "voiceover_text": voiceover,
            "subtitle_text": subtitle
        })

        timeline_lines.append(
            f"{format_social_timestamp(start_time)} - {subtitle}"
        )

        elapsed += duration

    calculated_duration = elapsed

    if not math.isfinite(total_duration) or total_duration <= 0:
        raise ValueError("Invalid total video duration")

    # Allow a small difference caused by media rounding.
    if abs(calculated_duration - total_duration) > 0.5:
        raise ValueError(
            f"Duration mismatch: scene total is "
            f"{calculated_duration:.2f}s, "
            f"reported total is {total_duration:.2f}s"
        )

    # --------------------------------------------------
    # 3. Build social media captions
    # --------------------------------------------------

    timeline = "\n".join(timeline_lines)

    caption_content = f"""=== SOCIAL MEDIA DELIVERABLES ===

[TITLE / HOOK]
{hook}

[VIDEO TOPIC]
{topic}

[YOUTUBE SHORTS DESCRIPTION]
{hook}

{caption}

TIMESTAMPS:
{timeline}

{hashtag_text} #Shorts #Technology

[INSTAGRAM REELS / TIKTOK CAPTION]
{hook}

{caption}

Share your thoughts in the comments.

{hashtag_text} #Reels #Technology
"""

    caption_file = output_dir / "caption.txt"
    temp_caption = output_dir / "caption_temp.txt"

    try:
        temp_caption.write_text(
            caption_content,
            encoding="utf-8"
        )
        temp_caption.replace(caption_file)
    finally:
        if temp_caption.exists():
            temp_caption.unlink()

    # --------------------------------------------------
    # 4. Record actual engine information
    # --------------------------------------------------

    actual_tts_engine = globals().get(
        "_LAST_TTS_ENGINE",
        "Not recorded"
    )

    if not actual_tts_engine:
        actual_tts_engine = "Not recorded"

    # --------------------------------------------------
    # 5. Generate metadata
    # --------------------------------------------------

    metadata = {
        "project": "Zero-API AI Video Factory",
        "video_title": hook,
        "selected_topic": topic,
        "target_audience": audience,
        "total_duration_seconds": round(calculated_duration, 3),
        "aspect_ratio": "9:16",
        "resolution": "720x1280",
        "frame_rate": 30,
        "visual_engine": engine_used,
        "llm_engine": "Qwen/Qwen2.5-7B-Instruct (4-bit NF4)",
        "tts_engine": actual_tts_engine,
        "audio_sample_rate_hz": 22050,
        "hashtags": hashtags,
        "scene_count": len(scenes),
        "scenes": scene_telemetry
    }

    metadata_file = output_dir / "video_metadata.json"
    temp_metadata = output_dir / "video_metadata_temp.json"

    try:
        temp_metadata.write_text(
            json.dumps(
                metadata,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )
        temp_metadata.replace(metadata_file)
    finally:
        if temp_metadata.exists():
            temp_metadata.unlink()

    return {
        "caption_path": str(caption_file),
        "metadata_path": str(metadata_file)
    }


print("✓ SEO Deliverables & Metadata Engine initialized.")


✓ SEO Deliverables & Metadata Engine initialized.


## CELL 12: Master Orchestrator, Preflight Diagnostic Smoke-Test & Execution
- **Smoke-Test Mode (`smoke_test_mode=True`):** Validates environment, Trends RSS, DDG news, Piper voice synthesis, Ken Burns motion, and subtitle rendering in ~10 seconds **without downloading heavy GPU models**.
- **Production Mode (`smoke_test_mode=False`):** Executes full 16-stage pipeline (Qwen 7B NF4 + Piper TTS + SDXL-Lightning 512×896 Ken Burns).
- Validates deliverable MP4 via `ffprobe` (720×1280 9:16, 25 FPS, H.264 video, AAC audio, duration ~50s).
- Embeds interactive HTML5 player to view the final render immediately in Colab.


In [ ]:
# ============================================================
# CELL 12: MASTER ORCHESTRATION & PRODUCTION EXECUTION
# ============================================================

import os
import json
import time
import math
import hashlib
import subprocess
from pathlib import Path
from typing import Any, Dict
from IPython.display import display, FileLink


# ------------------------------------------------------------
# 1. Generic model-field access
# ------------------------------------------------------------

def get_model_data(obj: Any) -> Dict[str, Any]:
    """Convert a Pydantic model or dictionary into a dictionary."""

    if isinstance(obj, dict):
        return obj

    if hasattr(obj, "model_dump"):
        return obj.model_dump()

    if hasattr(obj, "__dict__"):
        return vars(obj)

    return {}


def get_model_field(obj: Any, possible_names: list, default=None):
    """Read a field using supported aliases."""

    data = get_model_data(obj)

    for name in possible_names:
        value = data.get(name)
        if value is not None:
            return value

    for name in possible_names:
        try:
            value = getattr(obj, name, None)
            if value is not None:
                return value
        except Exception:
            pass

    return default


def get_video_plan_topic(plan: Any, fallback_topic: str = "") -> str:
    """Retrieve topic without assuming VideoPlan has a 'topic' field."""

    value = get_model_field(
        plan,
        ["topic", "selected_topic", "topic_title", "title", "video_topic"],
        default=""
    )

    if isinstance(value, str) and value.strip():
        return value.strip()

    return str(fallback_topic).strip()


def get_available_ram_gb():
    """Return available system RAM in GB without requiring psutil."""

    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return (pages * page_size) / (1024 ** 3)
    except Exception:
        return None


# ------------------------------------------------------------
# 2. Final MP4 validation
# ------------------------------------------------------------

def validate_final_mp4(mp4_path: str) -> Dict[str, Any]:

    path = Path(mp4_path)

    if not path.is_file() or path.stat().st_size < 1000:
        raise ValueError(f"Output file missing or invalid: {mp4_path}")

    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        str(path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"MP4 validation failed:\n{result.stderr[-3000:]}"
        )

    try:
        data = json.loads(result.stdout)
    except json.JSONDecodeError as exc:
        raise RuntimeError("FFprobe returned invalid JSON") from exc

    video_stream = next(
        (
            stream for stream in data.get("streams", [])
            if stream.get("codec_type") == "video"
        ),
        None
    )

    audio_stream = next(
        (
            stream for stream in data.get("streams", [])
            if stream.get("codec_type") == "audio"
        ),
        None
    )

    if not video_stream or not audio_stream:
        raise ValueError("Final MP4 must contain video and audio streams.")

    width = int(video_stream.get("width", 0))
    height = int(video_stream.get("height", 0))
    duration = float(data.get("format", {}).get("duration", 0.0))

    video_codec = video_stream.get("codec_name")
    audio_codec = audio_stream.get("codec_name")

    if width != 720 or height != 1280:
        raise ValueError(
            f"Incorrect resolution: {width}x{height}; expected 720x1280"
        )

    if abs((width / height) - (9 / 16)) > 0.001:
        raise ValueError("Incorrect video aspect ratio.")

    if duration <= 0 or not math.isfinite(duration):
        raise ValueError(f"Invalid video duration: {duration}")

    if video_codec != "h264":
        raise ValueError(f"Expected H.264, got {video_codec}")

    if audio_codec != "aac":
        raise ValueError(f"Expected AAC, got {audio_codec}")

    frame_rate_raw = (
        video_stream.get("avg_frame_rate")
        or video_stream.get("r_frame_rate")
        or "0/1"
    )

    try:
        numerator, denominator = frame_rate_raw.split("/")
        frame_rate = float(numerator) / float(denominator)
    except (ValueError, ZeroDivisionError):
        frame_rate = 0.0

    if not math.isfinite(frame_rate) or abs(frame_rate - 30.0) > 1.0:
        raise ValueError(
            f"Unexpected frame rate: {frame_rate:.2f} FPS"
        )

    report = {
        "valid": True,
        "width": width,
        "height": height,
        "aspect_ratio": "9:16",
        "frame_rate": round(frame_rate, 2),
        "duration_seconds": round(duration, 3),
        "video_codec": video_codec,
        "audio_codec": audio_codec,
        "file_size_mb": round(path.stat().st_size / (1024 ** 2), 2)
    }

    print("\n" + "=" * 60)
    print("           TECHNICAL VERIFICATION REPORT")
    print("=" * 60)
    print(f" Output Path     : {path}")
    print(f" File Size       : {report['file_size_mb']} MB")
    print(f" Duration        : {duration:.2f} seconds")
    print(f" Dimensions      : {width} x {height}")
    print(" Aspect Ratio    : 9:16")
    print(f" Frame Rate      : {frame_rate:.2f} FPS")
    print(f" Video Codec     : {video_codec.upper()}")
    print(f" Audio Codec     : {audio_codec.upper()}")
    print("=" * 60)

    return report


# ------------------------------------------------------------
# 3. Cache utilities
# ------------------------------------------------------------

def calculate_text_signature(value: Any) -> str:

    serialized = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def save_json_atomically(data: Dict[str, Any], destination: Path):

    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary = destination.with_name(
        destination.stem + "_temp" + destination.suffix
    )

    try:
        temporary.write_text(
            json.dumps(data, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8"
        )
        temporary.replace(destination)

    finally:
        if temporary.exists():
            temporary.unlink()


def save_video_plan(plan: Any, destination: Path):

    destination.parent.mkdir(parents=True, exist_ok=True)

    if hasattr(plan, "model_dump"):
        data = plan.model_dump()
    else:
        data = get_model_data(plan)

    save_json_atomically(data, destination)


# ------------------------------------------------------------
# 4. Topic-aware script generation and caching
# ------------------------------------------------------------

def load_or_generate_video_plan(brief: Dict[str, Any]):

    plan_cache = DIRS["scripts"] / "video_plan.json"
    context_cache = DIRS["scripts"] / "video_plan_context.json"

    brief_signature = calculate_text_signature(brief)

    selected_topic = str(
        brief.get("selected_topic", "")
    ).strip()

    if not selected_topic:
        raise ValueError(
            "Research brief does not contain selected_topic."
        )

    # Reuse cache when the research brief has not changed.
    # Do not require a 'topic' field in the VideoPlan schema.
    if plan_cache.exists() and context_cache.exists():

        try:
            cached_context = json.loads(
                context_cache.read_text(encoding="utf-8")
            )

            if cached_context.get("brief_signature") == brief_signature:

                cached_data = json.loads(
                    plan_cache.read_text(encoding="utf-8")
                )

                if hasattr(VideoPlan, "model_validate"):
                    cached_plan = VideoPlan.model_validate(cached_data)
                else:
                    cached_plan = VideoPlan.parse_obj(cached_data)

                cached_scenes = get_model_field(
                    cached_plan, ["scenes"], default=[]
                )

                if len(cached_scenes) == 10:
                    cached_topic = get_video_plan_topic(
                        cached_plan,
                        fallback_topic=selected_topic
                    )

                    print("  Reusing cached VideoPlan.")
                    print(f"  Topic: {cached_topic}")
                    return cached_plan

        except Exception as exc:
            print(f"  Cached plan rejected: {exc}")

    # Generate a new script.
    print("  Generating a fresh VideoPlan...")

    video_plan = generate_video_script_llm(brief)

    if video_plan is None:
        raise RuntimeError(
            "Script generation returned an empty VideoPlan."
        )

    plan_topic = get_video_plan_topic(
        video_plan,
        fallback_topic=selected_topic
    )

    scenes = get_model_field(
        video_plan,
        ["scenes"],
        default=[]
    )

    if len(scenes) != 10:
        raise ValueError(
            f"Expected 10 scenes, generated {len(scenes)}."
        )

    print(f"  VideoPlan topic: {plan_topic}")

    # Save the generated plan.
    save_video_plan(video_plan, plan_cache)

    save_json_atomically(
        {"brief_signature": brief_signature},
        context_cache
    )

    return video_plan


# ------------------------------------------------------------
# 5. Text-aware narration generation and caching
# ------------------------------------------------------------

def load_or_generate_scene_audio(scene: Any):

    scene_number = int(
        get_model_field(scene, ["scene_number"], default=0)
    )

    voiceover = str(
        get_model_field(
            scene,
            ["voiceover", "voiceover_text", "narration"],
            default=""
        )
    ).strip()

    if scene_number < 1:
        raise ValueError("Invalid scene number.")

    if not voiceover:
        raise ValueError(
            f"Scene {scene_number} has empty narration."
        )

    audio_path = (
        DIRS["audio"] / f"scene_{scene_number:02d}.wav"
    )

    signature_path = (
        DIRS["audio"] /
        f"scene_{scene_number:02d}_audio_signature.json"
    )

    # Include relevant TTS configuration in the cache signature.
    expected_signature = calculate_text_signature({
        "voiceover": voiceover,
        "tts_model": "en_US-lessac-medium",
        "sample_rate": 22050,
        "tts_speed": globals().get("TTS_SPEED", 1.0),
        "tts_engine": "piper"
    })

    reuse_audio = False

    if audio_path.exists() and signature_path.exists():

        try:
            cached_signature = json.loads(
                signature_path.read_text(encoding="utf-8")
            )

            if (
                cached_signature.get("signature") == expected_signature
                and audio_path.stat().st_size > 1000
            ):
                reuse_audio = True

        except Exception:
            reuse_audio = False

    if reuse_audio:
        print(f"  Scene {scene_number:02d}: Reusing narration.")

    else:
        print(f"  Scene {scene_number:02d}: Generating narration.")

        # Uses the synthesize_scene_voice function from Cell 7.
        synthesize_scene_voice(
            voiceover,
            str(audio_path)
        )

    if not audio_path.exists() or audio_path.stat().st_size <= 1000:
        raise RuntimeError(
            f"Narration generation failed for scene {scene_number}."
        )

    duration = get_audio_duration(str(audio_path))

    if not math.isfinite(duration) or duration <= 0:
        raise RuntimeError(
            f"Invalid narration duration for scene {scene_number}."
        )

    save_json_atomically(
        {"signature": expected_signature},
        signature_path
    )

    return str(audio_path), duration


# ------------------------------------------------------------
# 6. Main production orchestrator
# ------------------------------------------------------------

def run_video_factory(
    niche: str = "Artificial Intelligence and Robotics",
    geo: str = "IN",
    engine: str = "hybrid",
    background_music: bool = True
) -> Dict[str, Any]:

    start_time = time.time()

    print("\n" + "#" * 65)
    print("   ZERO-API AI SHORT-FORM VIDEO FACTORY")
    print("   PRODUCTION EXECUTION")
    print("#" * 65)

    # --------------------------------------------------------
    # Stage 1: Hardware diagnostics and RAM protection
    # --------------------------------------------------------

    print("\n>>> Stage 1/10: Hardware Diagnostics")

    diag = check_hardware_diagnostics()

    print(
        f"  Recommended engine: "
        f"{diag.get('recommended_engine', 'Not specified')}"
    )

    print(
        f"  Total GPU VRAM: "
        f"{diag.get('total_vram_gb', 0):.2f} GB"
    )

    available_ram = diag.get("available_ram_gb")

    if available_ram is None:
        available_ram = get_available_ram_gb()

    if available_ram is not None:
        print(f"  Available system RAM: {available_ram:.2f} GB")

        # Stop before loading Qwen or visual models if RAM is critically low.
        if available_ram < 3.0:
            raise MemoryError(
                f"Only {available_ram:.2f} GB system RAM is available. "
                "Stop execution, restart the Colab runtime, and rerun "
                "the notebook from the beginning. Do not load models "
                "with this amount of available RAM."
            )

    if engine not in ["hybrid", "wan", "sdxl", "procedural"]:
        raise ValueError(
            "Invalid engine. Choose hybrid, wan, sdxl, or procedural."
        )

    # --------------------------------------------------------
    # Stage 2: Trend discovery
    # --------------------------------------------------------

    print("\n>>> Stage 2/10: Trend Discovery")

    trends_ctx = fetch_live_trend_context(
        niche=niche,
        geo=geo
    )

    if not trends_ctx:
        raise RuntimeError(
            "Trend discovery returned no usable context."
        )

    # --------------------------------------------------------
    # Stage 3: Research and topic selection
    # --------------------------------------------------------

    print("\n>>> Stage 3/10: Topic Research")

    brief = evaluate_and_select_topic(
        trends_ctx,
        niche
    )

    if (
        not isinstance(brief, dict)
        or not brief.get("selected_topic")
    ):
        raise RuntimeError(
            "Topic research did not produce a valid topic."
        )

    print(f"  Selected topic: {brief['selected_topic']}")

    print(
        "  Evidence status: "
        f"{brief.get('evidence_status', 'Not specified')}"
    )

    # --------------------------------------------------------
    # Stage 4: Script generation
    # --------------------------------------------------------

    print("\n>>> Stage 4/10: Script Generation")

    video_plan = load_or_generate_video_plan(brief)

    scenes = get_model_field(
        video_plan,
        ["scenes"],
        default=[]
    )

    if len(scenes) != 10:
        raise ValueError(
            f"Expected 10 scenes, got {len(scenes)}."
        )

    plan_topic = get_video_plan_topic(
        video_plan,
        fallback_topic=brief["selected_topic"]
    )

    hook = get_model_field(
        video_plan,
        ["hook", "hook_title", "opening_hook"],
        default="Not specified"
    )

    print(f"  Topic: {plan_topic}")
    print(f"  Hook: {hook}")
    print(f"  Scenes: {len(scenes)}")

    # --------------------------------------------------------
    # Stage 5: Voice generation
    # --------------------------------------------------------

    print("\n>>> Stage 5/10: Narration Generation")

    audio_paths = []
    durations = []

    for scene in scenes:

        audio_path, duration = load_or_generate_scene_audio(scene)

        audio_paths.append(audio_path)
        durations.append(duration)

    total_narration_duration = sum(durations)

    print(
        f"  Total narration duration: "
        f"{total_narration_duration:.2f} seconds"
    )

    # --------------------------------------------------------
    # Stage 6: Visual generation
    # --------------------------------------------------------

    print("\n>>> Stage 6/10: Hybrid Visual Generation")

    # Cell 8 handles model loading, cache checks, RAM thresholds,
    # and procedural fallbacks.
    generate_visual_assets(
        video_plan,
        DIRS["images"],
        DIRS["scenes"],
        durations,
        engine=engine
    )

    clean_gpu_memory("Visual generation completed")

    # --------------------------------------------------------
    # Stage 7: Validate scene assets
    # --------------------------------------------------------

    print("\n>>> Stage 7/10: Scene Asset Validation")

    scene_clips = []

    for scene in scenes:

        scene_number = int(
            get_model_field(
                scene,
                ["scene_number"],
                default=0
            )
        )

        if scene_number < 1:
            raise ValueError("A scene has an invalid scene_number.")

        clip_path = (
            DIRS["scenes"] /
            f"scene_{scene_number:02d}.mp4"
        )

        if (
            not clip_path.exists()
            or clip_path.stat().st_size < 1000
        ):
            raise FileNotFoundError(
                f"Missing or invalid visual clip for "
                f"scene {scene_number}: {clip_path}"
            )

        scene_clips.append(str(clip_path))

    if len(scene_clips) != len(audio_paths):
        raise RuntimeError(
            "Scene video count does not match narration count."
        )

    # --------------------------------------------------------
    # Stage 8: Video assembly
    # --------------------------------------------------------

    print("\n>>> Stage 8/10: Video Assembly and Audio Ducking")

    subtitles = []

    for scene in scenes:

        subtitle = get_model_field(
            scene,
            ["subtitle", "subtitle_text"],
            default=""
        )

        if not str(subtitle).strip():
            raise ValueError(
                f"Missing subtitle for scene "
                f"{get_model_field(scene, ['scene_number'], '?')}."
            )

        subtitles.append(str(subtitle))

    final_mp4 = (
        DIRS["outputs"] / "final_video.mp4"
    )

    stitch_short_video(
        scene_clips,
        audio_paths,
        subtitles,
        str(final_mp4),
        DIRS["temp"],
        background_music=background_music
    )

    # --------------------------------------------------------
    # Stage 9: Final validation
    # --------------------------------------------------------

    print("\n>>> Stage 9/10: Final Validation")

    validation_report = validate_final_mp4(
        str(final_mp4)
    )

    # --------------------------------------------------------
    # Stage 10: SEO deliverables
    # --------------------------------------------------------

    print("\n>>> Stage 10/10: SEO Deliverables and Metadata")

    deliverables = generate_social_deliverables(
        video_plan,
        durations,
        validation_report["duration_seconds"],
        DIRS["outputs"],
        engine_used=engine
    )

    # --------------------------------------------------------
    # Production summary
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    print("\n" + "#" * 65)
    print("   PRODUCTION RUN COMPLETED")
    print("#" * 65)

    print(f" Topic       : {plan_topic}")
    print(f" Video       : {final_mp4}")
    print(f" Caption     : {deliverables['caption_path']}")
    print(f" Metadata    : {deliverables['metadata_path']}")

    print(
        f" Duration    : "
        f"{validation_report['duration_seconds']:.2f}s"
    )

    print(" Resolution  : 720x1280")

    print(
        f" Frame Rate  : "
        f"{validation_report['frame_rate']} FPS"
    )

    print(f" Elapsed     : {elapsed:.1f} seconds")
    print("#" * 65)

    return {
        "final_video": str(final_mp4),
        "caption": deliverables["caption_path"],
        "metadata": deliverables["metadata_path"],
        "validation": validation_report,
        "elapsed_seconds": round(elapsed, 2),
        "topic": plan_topic
    }


# ------------------------------------------------------------
# 7. Production configuration and execution
# ------------------------------------------------------------

production_result = run_video_factory(
    niche="Artificial Intelligence and Robotics",
    geo="IN",
    engine="hybrid",
    background_music=True
)


# ------------------------------------------------------------
# 8. Display final output
# ------------------------------------------------------------

if production_result.get("final_video"):

    final_video_path = Path(
        production_result["final_video"]
    ).resolve()

    if final_video_path.exists():

        print("\nFinal video is ready:")
        display(FileLink(str(final_video_path)))

        print("\nCaption:")
        display(FileLink(production_result["caption"]))

        print("\nMetadata:")
        display(FileLink(production_result["metadata"]))


#################################################################
   ZERO-API AI SHORT-FORM VIDEO FACTORY
   PRODUCTION EXECUTION
#################################################################

>>> Stage 1/10: Hardware Diagnostics
           COLAB HARDWARE & RUNTIME DIAGNOSTICS
 Python Version      : 3.13.16
 PyTorch Version     : 2.11.0+cu130
 CUDA Available      : True
 GPU Model           : Tesla T4
 Total VRAM          : 14.56 GB
 Available VRAM      : 14.46 GB
 CUDA Version        : 13.0
 Host RAM (Total)    : 12.67 GB
 Host RAM (Available): 4.38 GB
 Available Disk      : 112.09 GB
 Recommended Engine  : SDXL
  Recommended engine: sdxl
  Total GPU VRAM: 14.56 GB
  Available system RAM: 4.38 GB

>>> Stage 2/10: Trend Discovery

>>> Stage 3/10: Topic Research
✓ Research Brief compiled: This hilarious robot collapse reminds me that robot development is nowhere near...
  Evidence Status: Search-Retrieved News - Not Independently Verified
  Primary Source: TechRadar · via Yahoo Tec

/content/drive/MyDrive/VideoFactory/outputs/final_video.mp4


Caption:


/content/drive/MyDrive/VideoFactory/outputs/caption.txt


Metadata:


/content/drive/MyDrive/VideoFactory/outputs/video_metadata.json